# EU_Merger_Data_Quality_Check

In [2]:

import json
import os

file_path = "/content/case-data-M.json"

# Load the original JSON dataset
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Inspect the dataset structure
print("JSON loaded successfully")
print("Top-level data type:", type(data).__name__)
print("Number of case identifiers:", len(data))

# Inspect one sample record
case_id = next(iter(data))
case = data[case_id]

print("\nSample case ID:", case_id)
print("Record sections:", list(case.keys()))
print("Metadata field count:", len(case["metadata"]))
print("Number of decisions for sample:", len(case.get("decisions", [])))


JSON loaded successfully
Top-level data type: dict
Number of case identifiers: 10404

Sample case ID: M.3359
Record sections: ['metadata', 'caseAttachments', 'decisions']
Metadata field count: 28
Number of decisions for sample: 1


In [3]:

# Stage 2.2 — Task 1: Inspect JSON field names

sample_case = data["M.3359"]

# Case-level metadata
case_metadata = sample_case["metadata"]

print("=== CASE-LEVEL FIELDS ===")
for index, field in enumerate(case_metadata.keys(), start=1):
    print(f"{index}. {field}")

# Decision-level metadata
print("\n=== DECISION-LEVEL FIELDS ===")

decisions = sample_case.get("decisions", [])

if decisions:
    decision_metadata = decisions[0]["metadata"]

    for index, field in enumerate(decision_metadata.keys(), start=1):
        print(f"{index}. {field}")
else:
    print("No decisions available for this sample.")


=== CASE-LEVEL FIELDS ===
1. caseLastDecisionDate
2. caseExternalLinks
3. caseInvestigationPhase
4. caseLinks
5. caseInstrument
6. language
7. metadataReference
8. caseNumberPart
9. caseType
10. caseNumber
11. casePressReleases
12. caseSimplified
13. caseRegulation
14. caseTimelineEvents
15. caseDeadlineDate
16. caseInitiationDate
17. caseCourtCases
18. caseOfficialJournalPublications
19. caseRenotificationDate
20. caseCompanies
21. caseTitle
22. caseDg
23. caseOfficialJournalPublicationsPublishedDates
24. caseTimelineEventsDates
25. caseNotificationDate
26. casePressReleasesPublicationDates
27. caseSectors
28. metadataType

=== DECISION-LEVEL FIELDS ===
1. language
2. metadataReference
3. decisionExternalLinks
4. decisionOfficialJournalPublications
5. decisionPressReleases
6. decisionPressReleasesPublicationDates
7. decisionAdoptionDate
8. decisionTypes
9. decisionNumber
10. decisionOfficialJournalPublicationsPublishedDates
11. metadataType


In [4]:

# Stage 2.2 — Task 2
# Critical field completeness assessment

total_cases = len(data)

case_fields = [
    "caseNumber",
    "caseTitle",
    "caseNotificationDate",
    "caseLastDecisionDate",
    "caseInvestigationPhase",
    "caseSectors",
    "caseRegulation"
]

decision_fields = [
    "decisionAdoptionDate",
    "decisionTypes"
]

def has_value(value):
    if value is None or value == []:
        return False
    if isinstance(value, list):
        return any(
            v is not None and str(v).strip() != ""
            for v in value
        )
    return str(value).strip() != ""

print("=== CASE-LEVEL COMPLETENESS ===")

for field in case_fields:
    count = sum(
        has_value(record.get("metadata", {}).get(field))
        for record in data.values()
    )

    percentage = count / total_cases * 100

    print(f"{field}: {count:,} / {total_cases:,} "
          f"({percentage:.2f}%)")

print("\n=== DECISION-LEVEL COVERAGE ===")

for field in decision_fields:
    count = sum(
        any(
            has_value(
                decision.get("metadata", {}).get(field)
            )
            for decision in record.get("decisions", [])
        )
        for record in data.values()
    )

    percentage = count / total_cases * 100

    print(f"{field}: {count:,} / {total_cases:,} "
          f"({percentage:.2f}%)")

print("\n=== DECISION RECORD STRUCTURE ===")

no_decisions = sum(
    len(record.get("decisions", [])) == 0
    for record in data.values()
)

multiple_decisions = sum(
    len(record.get("decisions", [])) > 1
    for record in data.values()
)

print("Cases without decisions:", no_decisions)
print("Cases with multiple decisions:", multiple_decisions)


=== CASE-LEVEL COMPLETENESS ===
caseNumber: 10,404 / 10,404 (100.00%)
caseTitle: 10,404 / 10,404 (100.00%)
caseNotificationDate: 10,131 / 10,404 (97.38%)
caseLastDecisionDate: 10,364 / 10,404 (99.62%)
caseInvestigationPhase: 41 / 10,404 (0.39%)
caseSectors: 10,256 / 10,404 (98.58%)
caseRegulation: 10,404 / 10,404 (100.00%)

=== DECISION-LEVEL COVERAGE ===
decisionAdoptionDate: 10,364 / 10,404 (99.62%)
decisionTypes: 10,364 / 10,404 (99.62%)

=== DECISION RECORD STRUCTURE ===
Cases without decisions: 40
Cases with multiple decisions: 780


In [5]:

# Stage 2.2 — Task 3
# Inspect regulatory decision classifications

import json
from collections import Counter, defaultdict

decision_counts = Counter()
decision_cases = defaultdict(set)
phase_counts = Counter()

def decode_value(raw):
    if isinstance(raw, str):
        try:
            raw = json.loads(raw)
        except (ValueError, TypeError):
            return raw

    if isinstance(raw, dict):
        code = raw.get("code", "No code")
        label = raw.get("label", "No label")
        return f"{code} | {label}"

    return str(raw)

for case_id, record in data.items():

    metadata = record.get("metadata", {})

    # Inspect investigation phase values
    for value in metadata.get(
        "caseInvestigationPhase", []
    ):
        phase = decode_value(value)
        phase_counts[phase] += 1

    # Inspect all decision types
    for decision in record.get("decisions", []):
        decision_metadata = decision.get(
            "metadata", {}
        )

        for value in decision_metadata.get(
            "decisionTypes", []
        ):
            decision_type = decode_value(value)

            decision_counts[decision_type] += 1
            decision_cases[decision_type].add(case_id)

print("=== INVESTIGATION PHASE VALUES ===")

for phase, count in phase_counts.most_common():
    print(f"{phase}: {count:,}")

print("\n=== DECISION TYPE CLASSIFICATIONS ===")

for decision_type, count in decision_counts.most_common():
    unique_cases = len(decision_cases[decision_type])

    print(
        f"{decision_type} | "
        f"Entries: {count:,} | "
        f"Distinct cases: {unique_cases:,}"
    )

print("\n=== SUMMARY ===")
print("Distinct phase values:", len(phase_counts))
print("Distinct decision types:", len(decision_counts))


=== INVESTIGATION PHASE VALUES ===
FormalInvestigationPhase1Merger | 1: 37
FormalInvestigationPhase2Merger | 2: 4

=== DECISION TYPE CLASSIFICATIONS ===
DecisionType20310 | Art. 6(1)(b) | Entries: 7,030 | Distinct cases: 7,028
DecisionType310 | Art. 6(1)(b) | Entries: 2,146 | Distinct cases: 2,146
DecisionType20212 | Art. 6(1)(b) with conditions & obligations | Entries: 264 | Distinct cases: 263
DecisionType201220 | Art. 4(4) - Full referral | Entries: 223 | Distinct cases: 223
DecisionType20320 | Art. 6(1)(c) | Entries: 173 | Distinct cases: 173
DecisionType201316 | Art. 4(5) - Referral | Entries: 160 | Distinct cases: 160
DecisionType20490 | Withdrawn | Entries: 147 | Distinct cases: 145
DecisionType320 | Art. 6(1)(c) | Entries: 138 | Distinct cases: 136
DecisionType201250 | Purchaser approval | Entries: 118 | Distinct cases: 87
DecisionType20665 | Art. 10(3) | Entries: 110 | Distinct cases: 72
DecisionType440 | Art. 6(1)(b) with conditions & obligations | Entries: 107 | Distinct cas

In [6]:

# Stage 2.2 — Task 4
# Decision classifications by regulatory framework

import json
from collections import Counter, defaultdict

focus_labels = {
    "Art. 6(1)(c)",
    "Art. 6(1)(b) with conditions & obligations",
    "Art. 8(1)",
    "Art. 8(2)",
    "Art. 8(2) with conditions & obligations",
    "Art. 8(3)"
}

counts = Counter()
unique_cases = defaultdict(set)

for case_id, record in data.items():

    metadata = record.get("metadata", {})

    regulations = metadata.get("caseRegulation", [])
    regulation = (
        " / ".join(regulations)
        if regulations else "Missing"
    )

    for decision in record.get("decisions", []):

        decision_metadata = decision.get("metadata", {})

        for raw in decision_metadata.get("decisionTypes", []):

            if isinstance(raw, str):
                try:
                    item = json.loads(raw)
                except (ValueError, TypeError):
                    continue
            elif isinstance(raw, dict):
                item = raw
            else:
                continue

            if not isinstance(item, dict):
                continue

            label = item.get("label", "")

            if label in focus_labels:
                key = (regulation, label)
                counts[key] += 1
                unique_cases[key].add(case_id)

print("=== DECISION TYPES BY REGULATION ===")

for (regulation, label), count in sorted(counts.items()):

    case_count = len(unique_cases[(regulation, label)])

    print(
        f"{regulation} | {label} | "
        f"Entries: {count:,} | "
        f"Distinct cases: {case_count:,}"
    )


=== DECISION TYPES BY REGULATION ===
Council Regulation 139/2004 | Art. 6(1)(b) with conditions & obligations | Entries: 264 | Distinct cases: 263
Council Regulation 139/2004 | Art. 6(1)(c) | Entries: 174 | Distinct cases: 174
Council Regulation 139/2004 | Art. 8(1) | Entries: 40 | Distinct cases: 40
Council Regulation 139/2004 | Art. 8(2) with conditions & obligations | Entries: 82 | Distinct cases: 82
Council Regulation 139/2004 | Art. 8(3) | Entries: 14 | Distinct cases: 14
Council Regulation 4064/89 | Art. 6(1)(b) with conditions & obligations | Entries: 107 | Distinct cases: 104
Council Regulation 4064/89 | Art. 6(1)(c) | Entries: 137 | Distinct cases: 135
Council Regulation 4064/89 | Art. 8(2) | Entries: 27 | Distinct cases: 26
Council Regulation 4064/89 | Art. 8(2) with conditions & obligations | Entries: 72 | Distinct cases: 70
Council Regulation 4064/89 | Art. 8(3) | Entries: 19 | Distinct cases: 19


In [7]:

# Stage 2.2 — Task 5
# Validate Phase II case identification

import json

phase2_opened = set()
phase2_final = set()

final_labels = {
    "Art. 8(1)",
    "Art. 8(2)",
    "Art. 8(2) with conditions & obligations",
    "Art. 8(3)"
}

for case_id, record in data.items():

    labels = set()

    for decision in record.get("decisions", []):

        metadata = decision.get("metadata", {})

        for raw in metadata.get("decisionTypes", []):

            try:
                item = (
                    json.loads(raw)
                    if isinstance(raw, str)
                    else raw
                )
            except (ValueError, TypeError):
                continue

            if isinstance(item, dict):
                labels.add(item.get("label", ""))

    if "Art. 6(1)(c)" in labels:
        phase2_opened.add(case_id)

    if labels.intersection(final_labels):
        phase2_final.add(case_id)

both = phase2_opened & phase2_final
opened_only = phase2_opened - phase2_final
final_only = phase2_final - phase2_opened

print("=== PHASE II IDENTIFICATION ===")
print("Cases with Phase II initiation:", len(phase2_opened))
print("Cases with Article 8 final decisions:", len(phase2_final))
print("Cases containing both:", len(both))
print("Initiation recorded, no selected final decision:", len(opened_only))
print("Final decision recorded, no initiation:", len(final_only))

print("\n=== SAMPLE CASE IDS ===")
print("Initiation only:", sorted(opened_only)[:5])
print("Final decision only:", sorted(final_only)[:5])


=== PHASE II IDENTIFICATION ===
Cases with Phase II initiation: 309
Cases with Article 8 final decisions: 249
Cases containing both: 249
Initiation recorded, no selected final decision: 60
Final decision recorded, no initiation: 0

=== SAMPLE CASE IDS ===
Initiation only: ['M.10319', 'M.10325', 'M.1040', 'M.1044', 'M.1047']
Final decision only: []


In [8]:

# Stage 2.2 — Task 6
# Investigate 60 Phase II initiation-only cases

from collections import Counter
import json

other_decision_counts = Counter()
case_details = []

for case_id in sorted(opened_only):

    record = data[case_id]
    metadata = record.get("metadata", {})
    labels = set()

    for decision in record.get("decisions", []):

        decision_meta = decision.get("metadata", {})

        for raw in decision_meta.get("decisionTypes", []):

            try:
                item = (
                    json.loads(raw)
                    if isinstance(raw, str)
                    else raw
                )
            except (ValueError, TypeError):
                continue

            if isinstance(item, dict):
                label = item.get("label", "")
                if label:
                    labels.add(label)

    # Exclude the Phase II initiation itself
    other_labels = labels - {"Art. 6(1)(c)"}

    for label in other_labels:
        other_decision_counts[label] += 1

    if not other_labels:
        other_decision_counts[
            "No other decision classification"
        ] += 1

    title = metadata.get("caseTitle", ["Unknown"])
    title = title[0] if title else "Unknown"

    case_details.append(
        (case_id, title, sorted(other_labels))
    )

print("=== OTHER DECISION CLASSIFICATIONS ===")

for label, count in other_decision_counts.most_common():
    print(f"{label}: {count} cases")

print("\n=== SAMPLE CASE DETAILS ===")

for case_id, title, labels in case_details[:10]:
    print(f"\nCase: {case_id}")
    print(f"Title: {title}")
    print(f"Other decisions: {labels if labels else 'None recorded'}")

print("\n=== SUMMARY ===")
print("Cases investigated:", len(case_details))
print(
    "Cases with no other classification:",
    other_decision_counts[
        "No other decision classification"
    ]
)


=== OTHER DECISION CLASSIFICATIONS ===
Aborted / withdrawn: 34 cases
Aborted / withdrawn (N/2): 22 cases
Art. 18 (Statement of Objections): 20 cases
Art. 10(3): 8 cases
Art. 22 Full referral: 5 cases
Art. 5(2) - Incompleteness: 3 cases
No other decision classification: 2 cases
Art. 4(5) - Referral: 2 cases
Withdrawn: 1 cases
Art. 7(3) refusal: 1 cases

=== SAMPLE CASE DETAILS ===

Case: M.10319
Title: GREINER / RECTICEL
Other decisions: ['Aborted / withdrawn']

Case: M.10325
Title: KRONOSPAN / PFLEIDERER POLSKA
Other decisions: ['Aborted / withdrawn', 'Art. 10(3)']

Case: M.1040
Title: WOLTERS KLUWER / REED ELSEVIER
Other decisions: ['Aborted / withdrawn (N/2)', 'Art. 18 (Statement of Objections)']

Case: M.1044
Title: KPMG / ERNST & YOUNG
Other decisions: ['Aborted / withdrawn (N/2)']

Case: M.1047
Title: WIENERBERGER / CREMER & BREUER
Other decisions: ['Aborted / withdrawn (N/2)']

Case: M.10920
Title: AMAZON / IROBOT
Other decisions: ['Aborted / withdrawn', 'Art. 18 (Statement of Ob

In [9]:

# Stage 2.2 — Task 7
# Mutually exclusive classification of 60 cases

import json
from collections import defaultdict

withdrawal_labels = {
    "Aborted / withdrawn",
    "Aborted / withdrawn (N/2)",
    "Withdrawn"
}

referral_labels = {
    "Art. 22 Full referral",
    "Art. 4(5) - Referral",
    "Art. 4(4) - Full referral",
    "Art. 4(4) - Partial referral",
    "Art. 9(3) full referral",
    "Art. 9(3) partial referral"
}

groups = defaultdict(list)

for case_id in sorted(opened_only):
    labels = set()

    for decision in data[case_id].get("decisions", []):
        for raw in decision.get("metadata", {}).get(
            "decisionTypes", []
        ):
            try:
                item = (
                    json.loads(raw)
                    if isinstance(raw, str)
                    else raw
                )
            except (ValueError, TypeError):
                continue

            if isinstance(item, dict):
                labels.add(item.get("label", ""))

    labels.discard("Art. 6(1)(c)")
    labels.discard("")

    if labels & withdrawal_labels:
        category = "Withdrawal / abandonment recorded"
    elif labels & referral_labels:
        category = "Referral recorded"
    elif labels:
        category = "Other procedural classifications"
    else:
        category = "No additional classification"

    groups[category].append(case_id)

print("=== EXCLUSIVE CASE CLASSIFICATIONS ===")

categories = [
    "Withdrawal / abandonment recorded",
    "Referral recorded",
    "Other procedural classifications",
    "No additional classification"
]

for category in categories:
    print(f"{category}: {len(groups[category])}")

total = sum(len(groups[c]) for c in categories)
print("\nTotal classified:", total)
assert total == 60, "Classification total mismatch"

print("\n=== CASES REQUIRING FOLLOW-UP ===")

for category in categories[1:]:
    print(f"\n{category}:")
    print(groups[category])


=== EXCLUSIVE CASE CLASSIFICATIONS ===
Withdrawal / abandonment recorded: 57
Referral recorded: 0
Other procedural classifications: 1
No additional classification: 2

Total classified: 60

=== CASES REQUIRING FOLLOW-UP ===

Referral recorded:
[]

Other procedural classifications:
['M.12270']

No additional classification:
['M.11895', 'M.12236']


In [11]:

# Stage 2.2 — Task 8
# Investigate three cases requiring follow-up

import json

cases_to_check = [
    "M.12270",
    "M.11895",
    "M.12236"
]

def first_value(values):
    if isinstance(values, list):
        return values[0] if values else "Missing"
    return values if values is not None else "Missing"

for case_id in cases_to_check:

    record = data.get(case_id)

    if record is None:
        print(f"\nCase {case_id} not found")
        continue

    metadata = record.get("metadata", {})

    print("\n" + "=" * 55)
    print("CASE:", case_id)

    for field in [
        "caseTitle",
        "caseRegulation",
        "caseNotificationDate",
        "caseLastDecisionDate",
        "caseInvestigationPhase"
    ]:
        print(
            field + ":",
            first_value(metadata.get(field, []))
        )

    decisions = record.get("decisions", [])

    print("Number of decisions:", len(decisions))

    for index, decision in enumerate(decisions, 1):

        dm = decision.get("metadata", {})

        print(f"\nDecision {index}")
        print(
            "Adoption date:",
            first_value(
                dm.get("decisionAdoptionDate", [])
            )
        )

        for raw in dm.get("decisionTypes", []):

            try:
                item = (
                    json.loads(raw)
                    if isinstance(raw, str)
                    else raw
                )
            except (ValueError, TypeError):
                item = raw

            print("Decision type:", item)

        for attachment in decision.get(
            "decisionAttachments", []
        ):
            links = attachment.get(
                "metadata", {}
            ).get("attachmentLink", [])

            for link in links:
                print("Official document:", link)



CASE: M.12270
caseTitle: UPM-KYMMENE / SAPPI / JV
caseRegulation: Council Regulation 139/2004
caseNotificationDate: 2026-03-19
caseLastDecisionDate: 2026-10-05
caseInvestigationPhase: {"code":"FormalInvestigationPhase2Merger","label":"2"}
Number of decisions: 3

Decision 1
Adoption date: 2026-08-25
Decision type: {'code': 'DecisionType20545', 'label': 'Art. 18 (Statement of Objections)'}

Decision 2
Adoption date: 2026-10-05
Decision type: {'code': 'DecisionType20665', 'label': 'Art. 10(3)'}

Decision 3
Adoption date: 2026-04-28
Decision type: {'code': 'DecisionType20320', 'label': 'Art. 6(1)(c)'}

CASE: M.11895
caseTitle: XXXLUTZ / PORTA
caseRegulation: Council Regulation 139/2004
caseNotificationDate: 2026-09-07
caseLastDecisionDate: 2026-10-12
caseInvestigationPhase: {"code":"FormalInvestigationPhase2Merger","label":"2"}
Number of decisions: 1

Decision 1
Adoption date: 2026-10-12
Decision type: {'code': 'DecisionType20320', 'label': 'Art. 6(1)(c)'}

CASE: M.12236
caseTitle: SAIPEM

In [12]:

# Stage 2.3 — Task 1
# Build the initial case-level table

import pandas as pd

def first_value(metadata, field):
    values = metadata.get(field, [])
    return values[0] if values else None

rows = []

for case_id, record in data.items():
    metadata = record.get("metadata", {})

    rows.append({
        "case_id": case_id,
        "case_number": first_value(metadata, "caseNumber"),
        "case_title": first_value(metadata, "caseTitle"),
        "regulation": first_value(metadata, "caseRegulation"),
        "notification_date": first_value(
            metadata, "caseNotificationDate"
        ),
        "last_recorded_decision_date": first_value(
            metadata, "caseLastDecisionDate"
        ),
        "decision_count": len(
            record.get("decisions", [])
        ),
        "sector_count": len(
            metadata.get("caseSectors", [])
        )
    })

df_cases = pd.DataFrame(rows)

print("=== CASE TABLE VALIDATION ===")
print("Rows:", len(df_cases))
print("Columns:", len(df_cases.columns))
print("Duplicate case IDs:", df_cases["case_id"].duplicated().sum())
print("Case ID mismatches:",
      (df_cases["case_id"] != df_cases["case_number"]).sum())
print("Cases without decisions:",
      (df_cases["decision_count"] == 0).sum())
print("Cases with multiple decisions:",
      (df_cases["decision_count"] > 1).sum())

print("\n=== FIRST FIVE RECORDS ===")
display(df_cases.head())


=== CASE TABLE VALIDATION ===
Rows: 10404
Columns: 8
Duplicate case IDs: 0
Case ID mismatches: 0
Cases without decisions: 40
Cases with multiple decisions: 780

=== FIRST FIVE RECORDS ===


,case_id,case_number,case_title,regulation,notification_date,last_recorded_decision_date,decision_count,sector_count
0,M.3359,M.3359,ANGLO AMERICAN / BAUERNFEIND,Council Regulation 4064/89,2003-12-23,2004-02-04,1,1
1,M.2027,M.2027,DEUTSCHE BANK / SAP / JV,Council Regulation 4064/89,2000-06-09,2000-07-13,1,1
2,M.3358,M.3358,HOGG ROBINSON / KUONI,Council Regulation 4064/89,2003-12-18,2004-01-29,1,1
3,M.2028,M.2028,ABB / BILFINGER / MVV ENERGIE / JV,Council Regulation 4064/89,2000-06-22,2000-07-25,1,1
4,M.422,M.422,UNILEVER FRANCE / ORTIZ MIKO (II),Council Regulation 4064/89,1994-02-14,1994-03-15,1,1


In [13]:

# Stage 2.3 — Task 2
# Create the decision-level table

import pandas as pd
import json

decision_rows = []

for case_id, record in data.items():

    decisions = record.get("decisions", [])

    for decision in decisions:

        dm = decision.get("metadata", {})

        codes = []
        labels = []

        for raw in dm.get("decisionTypes", []):

            try:
                item = (
                    json.loads(raw)
                    if isinstance(raw, str)
                    else raw
                )
            except (ValueError, TypeError):
                continue

            if isinstance(item, dict):
                codes.append(item.get("code", ""))
                labels.append(item.get("label", ""))

        decision_rows.append({
            "case_id": case_id,
            "decision_id": first_value(
                dm, "metadataReference"
            ),
            "decision_number": first_value(
                dm, "decisionNumber"
            ),
            "decision_date": first_value(
                dm, "decisionAdoptionDate"
            ),
            "decision_type_codes": " | ".join(codes),
            "decision_type_labels": " | ".join(labels),
            "decision_type_count": len(codes)
        })

df_decisions = pd.DataFrame(decision_rows)

print("=== DECISION TABLE VALIDATION ===")

print("Total decision records:", len(df_decisions))

print("Columns:", len(df_decisions.columns))

print(
    "Unique cases represented:",
    df_decisions["case_id"].nunique()
)

print(
    "Duplicate case + decision IDs:",
    df_decisions.duplicated(
        subset=["case_id", "decision_id"]
    ).sum()
)

print(
    "Missing decision IDs:",
    df_decisions["decision_id"].isna().sum()
)

print(
    "Missing decision dates:",
    df_decisions["decision_date"].isna().sum()
)

print(
    "Decisions without classification:",
    (df_decisions["decision_type_count"] == 0).sum()
)

print(
    "Decisions with multiple classifications:",
    (df_decisions["decision_type_count"] > 1).sum()
)

print("\n=== FIRST FIVE DECISIONS ===")
display(df_decisions.head())



=== DECISION TABLE VALIDATION ===
Total decision records: 11625
Columns: 7
Unique cases represented: 10364
Duplicate case + decision IDs: 0
Missing decision IDs: 0
Missing decision dates: 0
Decisions without classification: 0
Decisions with multiple classifications: 0

=== FIRST FIVE DECISIONS ===


,case_id,decision_id,decision_number,decision_date,decision_type_codes,decision_type_labels,decision_type_count
0,M.3359,M.3359-DEC{DC119A65-A227-46C0-A154-CDFF388C7285},{DC119A65-A227-46C0-A154-CDFF388C7285},2004-02-04,DecisionType310,Art. 6(1)(b),1
1,M.2027,M.2027-DEC{DBCAB5A5-A161-468E-ACE4-F391CB523393},{DBCAB5A5-A161-468E-ACE4-F391CB523393},2000-07-13,DecisionType310,Art. 6(1)(b),1
2,M.3358,M.3358-DEC{140275C7-C111-4EAB-A394-02D0E3386C04},{140275C7-C111-4EAB-A394-02D0E3386C04},2004-01-29,DecisionType310,Art. 6(1)(b),1
3,M.2028,M.2028-DEC{993D4801-9582-4B6A-8D75-C2454BD8BE6C},{993D4801-9582-4B6A-8D75-C2454BD8BE6C},2000-07-25,DecisionType310,Art. 6(1)(b),1
4,M.422,M.422-DEC{CE63E0D3-AB54-4918-B602-CB0F6F218677},{CE63E0D3-AB54-4918-B602-CB0F6F218677},1994-03-15,DecisionType440,Art. 6(1)(b) with conditions & obligations,1


In [14]:

# Stage 2.3 — Task 3
# Create the case-sector relationship table

import pandas as pd
import json

sector_rows = []
parse_errors = []

for case_id, record in data.items():

    metadata = record.get("metadata", {})
    sectors = metadata.get("caseSectors", [])

    for raw in sectors:

        try:
            sector = (
                json.loads(raw)
                if isinstance(raw, str)
                else raw
            )

            if not isinstance(sector, dict):
                raise ValueError("Sector is not a dictionary")

            sector_rows.append({
                "case_id": case_id,
                "sector_code": sector.get("code"),
                "sector_label": sector.get("label")
            })

        except (ValueError, TypeError) as error:
            parse_errors.append((case_id, str(error)))

df_sectors = pd.DataFrame(
    sector_rows,
    columns=["case_id", "sector_code", "sector_label"]
)

print("=== SECTOR TABLE VALIDATION ===")

print("Total sector records:", len(df_sectors))
print("Columns:", len(df_sectors.columns))

print(
    "Unique cases represented:",
    df_sectors["case_id"].nunique()
)

print(
    "Unique sector codes:",
    df_sectors["sector_code"].nunique()
)

print(
    "Duplicate case + sector codes:",
    df_sectors.duplicated(
        subset=["case_id", "sector_code"]
    ).sum()
)

print(
    "Missing sector codes:",
    df_sectors["sector_code"].isna().sum()
)

print(
    "Missing sector labels:",
    df_sectors["sector_label"].isna().sum()
)

sector_counts = df_sectors.groupby("case_id").size()

print(
    "Cases with multiple sectors:",
    (sector_counts > 1).sum()
)

print(
    "Cases without sectors:",
    len(data) - df_sectors["case_id"].nunique()
)

print("JSON parsing errors:", len(parse_errors))

print("\n=== FIRST FIVE SECTOR RECORDS ===")
display(df_sectors.head())


=== SECTOR TABLE VALIDATION ===
Total sector records: 16310
Columns: 3
Unique cases represented: 10256
Unique sector codes: 1313
Duplicate case + sector codes: 0
Missing sector codes: 0
Missing sector labels: 0
Cases with multiple sectors: 3452
Cases without sectors: 148
JSON parsing errors: 0

=== FIRST FIVE SECTOR RECORDS ===


,case_id,sector_code,sector_label
0,M.3359,NaceSectorsC_17_1,"C.17.1 - Manufacture of pulp, paper and paperb..."
1,M.2027,NaceSectorsG_46,"G.46 - Wholesale trade, except of motor vehicl..."
2,M.3358,NaceSectorsN_79_1_2,N.79.12 - Tour operator activities
3,M.2028,NaceSectorsE_36_0_0,"E.36.00 - Water collection, treatment and supply"
4,M.422,NaceSectorsC_10_5_2,C.10.52 - Manufacture of ice cream


In [15]:

# Stage 2.3 — Task 4
# Validate relational integrity and recorded counts

case_ids = set(df_cases["case_id"])
decision_ids = set(df_decisions["case_id"])
sector_ids = set(df_sectors["case_id"])

# Check for orphan records
orphan_decisions = decision_ids - case_ids
orphan_sectors = sector_ids - case_ids

# Compare recorded counts against extracted tables
actual_decision_counts = (
    df_decisions.groupby("case_id").size()
    .reindex(df_cases["case_id"], fill_value=0)
    .to_numpy()
)

actual_sector_counts = (
    df_sectors.groupby("case_id").size()
    .reindex(df_cases["case_id"], fill_value=0)
    .to_numpy()
)

decision_mismatches = (
    df_cases["decision_count"].to_numpy()
    != actual_decision_counts
).sum()

sector_mismatches = (
    df_cases["sector_count"].to_numpy()
    != actual_sector_counts
).sum()

# Check whether sector codes map consistently to labels
inconsistent_sector_labels = (
    df_sectors.groupby("sector_code")["sector_label"]
    .nunique()
    .gt(1)
    .sum()
)

print("=== RELATIONAL INTEGRITY VALIDATION ===")

print("Unique case IDs:", len(case_ids))
print("Orphan decision case IDs:", len(orphan_decisions))
print("Orphan sector case IDs:", len(orphan_sectors))
print("Decision count mismatches:", decision_mismatches)
print("Sector count mismatches:", sector_mismatches)
print("Sector codes with inconsistent labels:",
      inconsistent_sector_labels)

print("\n=== TABLE CONSISTENCY ===")

print("Cases in df_cases:", len(df_cases))
print("Decision rows:", len(df_decisions))
print("Sector rows:", len(df_sectors))

print("\n=== OVERALL CHECK ===")

if (
    len(orphan_decisions) == 0
    and len(orphan_sectors) == 0
    and decision_mismatches == 0
    and sector_mismatches == 0
    and inconsistent_sector_labels == 0
):
    print("PASS: All relational checks successful")
else:
    print("REVIEW REQUIRED: Investigate validation issues")


=== RELATIONAL INTEGRITY VALIDATION ===
Unique case IDs: 10404
Orphan decision case IDs: 0
Orphan sector case IDs: 0
Decision count mismatches: 0
Sector count mismatches: 0
Sector codes with inconsistent labels: 0

=== TABLE CONSISTENCY ===
Cases in df_cases: 10404
Decision rows: 11625
Sector rows: 16310

=== OVERALL CHECK ===
PASS: All relational checks successful


In [16]:

# ==================================================
# Stage 2.4 — Task 1
# Regulatory Date Quality and Consistency Validation
# ==================================================

import pandas as pd

# Preserve original dataframes
cases_check = df_cases.copy()
decisions_check = df_decisions.copy()

# Convert date strings into datetime values
cases_check["notification_dt"] = pd.to_datetime(
    cases_check["notification_date"],
    errors="coerce"
)

cases_check["last_decision_dt"] = pd.to_datetime(
    cases_check["last_recorded_decision_date"],
    errors="coerce"
)

decisions_check["decision_dt"] = pd.to_datetime(
    decisions_check["decision_date"],
    errors="coerce"
)

# 1. Check missing and invalid dates
print("=== DATE COMPLETENESS ===")

print(
    "Missing notification dates:",
    cases_check["notification_date"].isna().sum()
)

print(
    "Invalid notification dates:",
    (
        cases_check["notification_date"].notna()
        & cases_check["notification_dt"].isna()
    ).sum()
)

print(
    "Missing last decision dates:",
    cases_check[
        "last_recorded_decision_date"
    ].isna().sum()
)

print(
    "Invalid last decision dates:",
    (
        cases_check[
            "last_recorded_decision_date"
        ].notna()
        & cases_check["last_decision_dt"].isna()
    ).sum()
)

print(
    "Missing or invalid individual decision dates:",
    decisions_check["decision_dt"].isna().sum()
)

# 2. Check chronological consistency
both_dates = (
    cases_check["notification_dt"].notna()
    & cases_check["last_decision_dt"].notna()
)

negative_sequence = (
    both_dates
    & (
        cases_check["last_decision_dt"]
        < cases_check["notification_dt"]
    )
)

print("\n=== CHRONOLOGICAL VALIDATION ===")

print("Cases with both dates:", both_dates.sum())

print(
    "Last decision before notification:",
    negative_sequence.sum()
)

# 3. Compare case dates with decision table
latest_decisions = (
    decisions_check.groupby("case_id")["decision_dt"]
    .max()
    .rename("max_decision_dt")
)

date_comparison = cases_check[
    ["case_id", "last_decision_dt"]
].merge(
    latest_decisions,
    on="case_id",
    how="left"
)

both_recorded = (
    date_comparison["last_decision_dt"].notna()
    & date_comparison["max_decision_dt"].notna()
)

date_mismatch = (
    both_recorded
    & (
        date_comparison["last_decision_dt"]
        != date_comparison["max_decision_dt"]
    )
)

print("\n=== CROSS-TABLE DATE VALIDATION ===")

print(
    "Cases with dates in both tables:",
    both_recorded.sum()
)

print(
    "Last decision date mismatches:",
    date_mismatch.sum()
)

# 4. Display sample anomalies
print("\n=== SAMPLE DATE ISSUES ===")

display(
    date_comparison.loc[date_mismatch].head(10)
)

# 5. Determine historical coverage
print("\n=== DATE RANGES ===")

print(
    "Earliest notification:",
    cases_check["notification_dt"].min()
)

print(
    "Latest notification:",
    cases_check["notification_dt"].max()
)

print(
    "Earliest recorded decision:",
    decisions_check["decision_dt"].min()
)

print(
    "Latest recorded decision:",
    decisions_check["decision_dt"].max()
)


=== DATE COMPLETENESS ===
Missing notification dates: 273
Invalid notification dates: 0
Missing last decision dates: 40
Invalid last decision dates: 0
Missing or invalid individual decision dates: 0

=== CHRONOLOGICAL VALIDATION ===
Cases with both dates: 10100
Last decision before notification: 1

=== CROSS-TABLE DATE VALIDATION ===
Cases with dates in both tables: 10364
Last decision date mismatches: 98

=== SAMPLE DATE ISSUES ===


,case_id,last_decision_dt,max_decision_dt
2083,M.3535,2004-12-07,2004-12-08
2179,M.1383,1999-09-28,1999-09-29
3485,M.6800,2015-06-15,2015-06-16
4784,M.4439,2007-10-10,2007-10-11
5445,M.7716,2015-09-08,2015-09-09
5486,M.7750,2015-10-27,2015-10-28
5546,M.7300,2014-09-11,2014-09-12
5592,M.7314,2014-09-01,2014-09-02
5598,M.8654,2017-12-19,2017-12-20
5635,M.8668,2017-11-23,2017-11-24



=== DATE RANGES ===
Earliest notification: 1990-10-04 00:00:00
Latest notification: 2026-10-29 00:00:00
Earliest recorded decision: 1990-11-07 00:00:00
Latest recorded decision: 2026-10-29 00:00:00


In [17]:

# Stage 2.4 — Task 2
# Investigate chronological and date discrepancies

# 1. Inspect the case with a negative date sequence
print("=== NEGATIVE DATE CASE ===")

negative_cases = cases_check.loc[
    negative_sequence,
    [
        "case_id",
        "case_title",
        "regulation",
        "notification_dt",
        "last_decision_dt"
    ]
].copy()

display(negative_cases)

# 2. Calculate differences between recorded dates
print("\n=== DATE MISMATCH DISTRIBUTION ===")

mismatches = date_comparison.loc[
    date_mismatch,
    [
        "case_id",
        "last_decision_dt",
        "max_decision_dt"
    ]
].copy()

mismatches["difference_days"] = (
    mismatches["max_decision_dt"]
    - mismatches["last_decision_dt"]
).dt.days

print("Total mismatches:", len(mismatches))

print("\nDifference in calendar days:")
print(
    mismatches["difference_days"]
    .value_counts()
    .sort_index()
)

print(
    "\nDifferences greater than one day:",
    (mismatches["difference_days"].abs() > 1).sum()
)

# 3. Show the largest discrepancies
print("\n=== LARGEST DATE DISCREPANCIES ===")

display(
    mismatches.reindex(
        mismatches["difference_days"]
        .abs()
        .sort_values(ascending=False)
        .index
    ).head(10)
)


=== NEGATIVE DATE CASE ===


,case_id,case_title,regulation,notification_dt,last_decision_dt
10395,M.12355,BOSTON SCIENTIFIC / PENUMBRA,Council Regulation 139/2004,2026-10-29,2026-05-07



=== DATE MISMATCH DISTRIBUTION ===
Total mismatches: 98

Difference in calendar days:
difference_days
-30     1
 1     97
Name: count, dtype: int64

Differences greater than one day: 1

=== LARGEST DATE DISCREPANCIES ===


,case_id,last_decision_dt,max_decision_dt,difference_days
7714,M.9142,2019-08-07,2019-07-08,-30
2083,M.3535,2004-12-07,2004-12-08,1
3485,M.6800,2015-06-15,2015-06-16,1
4784,M.4439,2007-10-10,2007-10-11,1
5445,M.7716,2015-09-08,2015-09-09,1
5486,M.7750,2015-10-27,2015-10-28,1
5546,M.7300,2014-09-11,2014-09-12,1
5592,M.7314,2014-09-01,2014-09-02,1
5598,M.8654,2017-12-19,2017-12-20,1
5635,M.8668,2017-11-23,2017-11-24,1


In [18]:

# Stage 2.4 — Task 3
# Inspect exceptional regulatory date records

import json

cases_to_investigate = ["M.12355", "M.9142"]

for case_id in cases_to_investigate:

    record = data[case_id]
    metadata = record.get("metadata", {})

    print("\n" + "=" * 55)
    print("CASE:", case_id)

    for field in [
        "caseTitle",
        "caseRegulation",
        "caseInitiationDate",
        "caseNotificationDate",
        "caseRenotificationDate",
        "caseLastDecisionDate"
    ]:
        print(field, ":", metadata.get(field, []))

    print("\n=== RECORDED DECISIONS ===")

    for decision in record.get("decisions", []):
        dm = decision.get("metadata", {})

        print("\nDecision ID:",
              dm.get("metadataReference", []))

        print("Adoption date:",
              dm.get("decisionAdoptionDate", []))

        for raw in dm.get("decisionTypes", []):
            item = json.loads(raw) if isinstance(raw, str) else raw
            print("Decision type:", item)



CASE: M.12355
caseTitle : ['BOSTON SCIENTIFIC / PENUMBRA']
caseRegulation : ['Council Regulation 139/2004']
caseInitiationDate : ['2026-02-09']
caseNotificationDate : ['2026-10-29']
caseRenotificationDate : []
caseLastDecisionDate : ['2026-05-07']

=== RECORDED DECISIONS ===

Decision ID: ['M.12355-DEC{E3410D12-AA01-43D7-8F6F-D709ABE254A9}']
Adoption date: ['2026-05-07']
Decision type: {'code': 'DecisionType201316', 'label': 'Art. 4(5) - Referral'}

CASE: M.9142
caseTitle : ['REWE / LEKKERLAND']
caseRegulation : ['Council Regulation 139/2004']
caseInitiationDate : ['2018-09-25']
caseNotificationDate : ['2019-07-15']
caseRenotificationDate : []
caseLastDecisionDate : ['2019-08-07']

=== RECORDED DECISIONS ===

Decision ID: ['M.9142-DEC{BEF5BC35-743D-4D67-9D7E-78A330D05AD2}']
Adoption date: ['2019-07-08']
Decision type: {'code': 'DecisionType201203', 'label': 'Art. 4(4) - Partial referral'}


In [19]:

# Stage 2.4 — Task 4
# Audit decisions recorded before notification

import pandas as pd

# Join decisions to their parent merger case
date_audit = df_decisions.merge(
    df_cases[
        ["case_id", "case_title", "notification_date"]
    ],
    on="case_id",
    how="left",
    validate="many_to_one"
)

# Convert dates
date_audit["notification_dt"] = pd.to_datetime(
    date_audit["notification_date"],
    errors="coerce"
)

date_audit["decision_dt"] = pd.to_datetime(
    date_audit["decision_date"],
    errors="coerce"
)

# Calculate relative timing
date_audit["days_from_notification"] = (
    date_audit["decision_dt"]
    - date_audit["notification_dt"]
).dt.days

# Identify decisions before notification
pre_notification = date_audit[
    date_audit["days_from_notification"] < 0
].copy()

print("=== PRE-NOTIFICATION DECISION AUDIT ===")

print("Total decision records:", len(date_audit))

print(
    "Decision records before notification:",
    len(pre_notification)
)

print(
    "Unique cases with pre-notification decisions:",
    pre_notification["case_id"].nunique()
)

print(
    "Decisions with missing notification dates:",
    date_audit["notification_dt"].isna().sum()
)

print("\n=== PRE-NOTIFICATION DECISION TYPES ===")

print(
    pre_notification["decision_type_labels"]
    .value_counts()
    .to_string()
)

print("\n=== TWO VERIFIED EXAMPLE CASES ===")

display(
    date_audit[
        date_audit["case_id"].isin(
            ["M.12355", "M.9142"]
        )
    ][[
        "case_id",
        "case_title",
        "notification_date",
        "decision_date",
        "decision_type_labels",
        "days_from_notification"
    ]]
)

print("\n=== SAMPLE PRE-NOTIFICATION RECORDS ===")

display(
    pre_notification[[
        "case_id",
        "notification_date",
        "decision_date",
        "decision_type_labels",
        "days_from_notification"
    ]].head(10)
)


=== PRE-NOTIFICATION DECISION AUDIT ===
Total decision records: 11625
Decision records before notification: 278
Unique cases with pre-notification decisions: 273
Decisions with missing notification dates: 287

=== PRE-NOTIFICATION DECISION TYPES ===
decision_type_labels
Art. 4(5) - Referral                 152
Art. 7(3)                             40
Art. 22 Full referral                 37
Art. 4(4) - Partial referral          23
Art. 5(2) - Incompleteness            20
Art. 6(1)(c)                           1
Art. 18 (Statement of Objections)      1
Art. 8(2)                              1
Art. 22 Refusal of referral            1
Art. 4(4) - No referral                1
Rejection of a request to act          1

=== TWO VERIFIED EXAMPLE CASES ===


,case_id,case_title,notification_date,decision_date,decision_type_labels,days_from_notification
8679,M.9142,REWE / LEKKERLAND,2019-07-15,2019-07-08,Art. 4(4) - Partial referral,-7.0
11616,M.12355,BOSTON SCIENTIFIC / PENUMBRA,2026-10-29,2026-05-07,Art. 4(5) - Referral,-175.0



=== SAMPLE PRE-NOTIFICATION RECORDS ===


,case_id,notification_date,decision_date,decision_type_labels,days_from_notification
328,M.3709,2005-01-31,2005-01-20,Art. 7(3),-11.0
480,M.3754,2005-04-29,2005-03-02,Art. 7(3),-58.0
690,M.3796,2005-08-04,2005-05-18,Art. 22 Full referral,-78.0
767,M.3333,2007-01-31,2004-02-12,Art. 6(1)(c),-1084.0
768,M.3333,2007-01-31,2004-07-19,Art. 8(2),-926.0
771,M.3333,2007-01-31,2004-05-24,Art. 18 (Statement of Objections),-982.0
1023,M.3275,2004-10-01,2004-09-14,Art. 7(3),-17.0
1377,M.3637,2005-02-17,2005-02-02,Art. 5(2) - Incompleteness,-15.0
1488,M.3664,2005-01-13,2005-01-06,Art. 5(2) - Incompleteness,-7.0
1585,M.3686,2005-02-07,2005-02-04,Art. 5(2) - Incompleteness,-3.0


In [20]:

# Stage 2.4 — Task 5
# Investigate unusual pre-notification decisions

unusual_labels = [
    "Art. 6(1)(c)",
    "Art. 18 (Statement of Objections)",
    "Art. 8(2)",
    "Art. 22 Refusal of referral",
    "Art. 4(4) - No referral",
    "Rejection of a request to act"
]

unusual_records = pre_notification[
    pre_notification["decision_type_labels"].isin(
        unusual_labels
    )
].copy()

print("=== UNUSUAL PRE-NOTIFICATION RECORDS ===")

display(
    unusual_records[[
        "case_id",
        "case_title",
        "notification_date",
        "decision_date",
        "decision_type_labels",
        "days_from_notification"
    ]].sort_values("case_id")
)

print("\n=== CASE METADATA INVESTIGATION ===")

for case_id in sorted(unusual_records["case_id"].unique()):

    metadata = data[case_id].get("metadata", {})

    print("\n" + "=" * 55)
    print("Case:", case_id)

    for field in [
        "caseTitle",
        "caseRegulation",
        "caseInitiationDate",
        "caseNotificationDate",
        "caseRenotificationDate",
        "caseLastDecisionDate",
        "caseLinks",
        "caseTimelineEventsDates"
    ]:
        print(
            field + ":",
            metadata.get(field, [])
        )

print("\n=== SUMMARY ===")

print("Unusual decision records:", len(unusual_records))
print(
    "Unique cases requiring investigation:",
    unusual_records["case_id"].nunique()
)


=== UNUSUAL PRE-NOTIFICATION RECORDS ===


,case_id,case_title,notification_date,decision_date,decision_type_labels,days_from_notification
8951,M.10438,MOL / OMV SLOVENIJA,2022-05-13,2021-10-28,Art. 4(4) - No referral,-197.0
767,M.3333,SONY / BMG (4064),2007-01-31,2004-02-12,Art. 6(1)(c),-1084.0
768,M.3333,SONY / BMG (4064),2007-01-31,2004-07-19,Art. 8(2),-926.0
771,M.3333,SONY / BMG (4064),2007-01-31,2004-05-24,Art. 18 (Statement of Objections),-982.0
4544,M.5828,PROCTER & GAMBLE / SARA LEE Air care,2010-05-10,2010-03-31,Art. 22 Refusal of referral,-40.0
8991,M.9969,VEOLIA / SUEZ,2021-10-22,2020-12-17,Rejection of a request to act,-309.0



=== CASE METADATA INVESTIGATION ===

Case: M.10438
caseTitle: ['MOL / OMV SLOVENIJA']
caseRegulation: ['Council Regulation 139/2004']
caseInitiationDate: ['2021-08-03']
caseNotificationDate: ['2022-05-13']
caseRenotificationDate: []
caseLastDecisionDate: ['2023-05-17']
caseLinks: []
caseTimelineEventsDates: ['2023-03-17', '2022-07-11', '2023-03-17']

Case: M.3333
caseTitle: ['SONY / BMG (4064)']
caseRegulation: ['Council Regulation 4064/89']
caseInitiationDate: ['2003-11-07']
caseNotificationDate: ['2007-01-31']
caseRenotificationDate: []
caseLastDecisionDate: ['2007-10-03']
caseLinks: []
caseTimelineEventsDates: ['2007-06-25', '2007-03-19']

Case: M.5828
caseTitle: ['PROCTER & GAMBLE / SARA LEE Air care']
caseRegulation: ['Council Regulation 139/2004']
caseInitiationDate: ['2010-02-22']
caseNotificationDate: ['2010-05-10']
caseRenotificationDate: []
caseLastDecisionDate: ['2010-06-17']
caseLinks: []
caseTimelineEventsDates: []

Case: M.9969
caseTitle: ['VEOLIA / SUEZ']
caseRegulation

In [21]:

# Stage 2.4 — Task 6
# Build a reusable regulatory date audit

import pandas as pd
import numpy as np

df_date_audit = date_audit.copy()

# Classify the chronological relationship
df_date_audit["timing_status"] = np.select(
    [
        df_date_audit["notification_dt"].isna(),
        df_date_audit["days_from_notification"] < 0
    ],
    [
        "Notification date missing",
        "Decision before recorded notification"
    ],
    default="Decision on/after notification"
)

# Record verified explanations for selected exceptions
exception_notes = {
    "M.3333": "Historical review episodes require separation",
    "M.10438": "Pre-notification no-referral procedure",
    "M.5828": "Referral procedure before EU notification",
    "M.9969": "Pre-notification procedural request"
}

df_date_audit["exception_note"] = pd.NA

before_notification = (
    df_date_audit["days_from_notification"] < 0
)

for case_id, note in exception_notes.items():
    mask = (
        df_date_audit["case_id"].eq(case_id)
        & before_notification
    )
    df_date_audit.loc[mask, "exception_note"] = note

print("=== DATE AUDIT SUMMARY ===")
print(df_date_audit["timing_status"].value_counts())

print("\n=== TOTAL VALIDATION ===")
print("Audited decisions:", len(df_date_audit))
print("Original decisions:", len(df_decisions))

assert len(df_date_audit) == len(df_decisions)
print("PASS: All decision records accounted for")

print("\n=== DOCUMENTED EXCEPTIONS ===")

display(
    df_date_audit[
        df_date_audit["exception_note"].notna()
    ][[
        "case_id",
        "decision_date",
        "notification_date",
        "decision_type_labels",
        "exception_note"
    ]].sort_values(["case_id", "decision_date"])
)


=== DATE AUDIT SUMMARY ===
timing_status
Decision on/after notification           11060
Notification date missing                  287
Decision before recorded notification      278
Name: count, dtype: int64

=== TOTAL VALIDATION ===
Audited decisions: 11625
Original decisions: 11625
PASS: All decision records accounted for

=== DOCUMENTED EXCEPTIONS ===


,case_id,decision_date,notification_date,decision_type_labels,exception_note
8951,M.10438,2021-10-28,2022-05-13,Art. 4(4) - No referral,Pre-notification no-referral procedure
767,M.3333,2004-02-12,2007-01-31,Art. 6(1)(c),Historical review episodes require separation
771,M.3333,2004-05-24,2007-01-31,Art. 18 (Statement of Objections),Historical review episodes require separation
768,M.3333,2004-07-19,2007-01-31,Art. 8(2),Historical review episodes require separation
4544,M.5828,2010-03-31,2010-05-10,Art. 22 Refusal of referral,Referral procedure before EU notification
4545,M.5828,2010-03-31,2010-05-10,Art. 22 Full referral,Referral procedure before EU notification
8991,M.9969,2020-12-17,2021-10-22,Rejection of a request to act,Pre-notification procedural request


In [22]:

# Stage 2.4 — Task 7
# Audit the 97 one-day date discrepancies

import pandas as pd

# Use the mismatch table created in Task 2
one_day = mismatches.loc[
    mismatches["difference_days"] == 1
].copy()

# Add regulatory information
one_day = one_day.merge(
    df_cases[["case_id", "regulation"]],
    on="case_id",
    how="left",
    validate="one_to_one"
)

one_day["decade"] = (
    one_day["max_decision_dt"].dt.year // 10 * 10
)

print("=== ONE-DAY DISCREPANCY SUMMARY ===")
print("Cases with +1 day difference:", len(one_day))
print("Unique cases:", one_day["case_id"].nunique())

print("\n=== BY REGULATION ===")
print(one_day["regulation"].value_counts().to_string())

print("\n=== BY DECADE ===")
print(
    one_day["decade"]
    .value_counts()
    .sort_index()
    .to_string()
)

# Identify the decision classifications associated
# with the later date in the decision table
latest_records = df_decisions.copy()

latest_records["decision_dt"] = pd.to_datetime(
    latest_records["decision_date"],
    errors="coerce"
)

latest_records = latest_records.merge(
    one_day[["case_id", "max_decision_dt"]],
    on="case_id",
    how="inner"
)

latest_records = latest_records[
    latest_records["decision_dt"]
    == latest_records["max_decision_dt"]
]

print("\n=== LATEST DECISION CLASSIFICATIONS ===")
print(
    latest_records["decision_type_labels"]
    .value_counts()
    .to_string()
)

print("\n=== SAMPLE CASES ===")
display(
    one_day[[
        "case_id",
        "regulation",
        "last_decision_dt",
        "max_decision_dt",
        "difference_days"
    ]].head(10)
)


=== ONE-DAY DISCREPANCY SUMMARY ===
Cases with +1 day difference: 97
Unique cases: 97

=== BY REGULATION ===
regulation
Council Regulation 139/2004    96
Council Regulation 4064/89      1

=== BY DECADE ===
decade
1990     1
2000     2
2010    80
2020    14

=== LATEST DECISION CLASSIFICATIONS ===
decision_type_labels
Art. 6(1)(b)                                  75
Purchaser approval                            10
Art. 8(2) with conditions & obligations        2
Aborted / withdrawn                            2
Art. 8(1)                                      2
Art. 6(1)(b) with conditions & obligations     2
Withdrawn                                      1
No substance                                   1
Decision under remedy review clause            1
Art. 18 (Statement of Objections)              1

=== SAMPLE CASES ===


,case_id,regulation,last_decision_dt,max_decision_dt,difference_days
0,M.3535,Council Regulation 139/2004,2004-12-07,2004-12-08,1
1,M.1383,Council Regulation 4064/89,1999-09-28,1999-09-29,1
2,M.6800,Council Regulation 139/2004,2015-06-15,2015-06-16,1
3,M.4439,Council Regulation 139/2004,2007-10-10,2007-10-11,1
4,M.7716,Council Regulation 139/2004,2015-09-08,2015-09-09,1
5,M.7750,Council Regulation 139/2004,2015-10-27,2015-10-28,1
6,M.7300,Council Regulation 139/2004,2014-09-11,2014-09-12,1
7,M.7314,Council Regulation 139/2004,2014-09-01,2014-09-02,1
8,M.8654,Council Regulation 139/2004,2017-12-19,2017-12-20,1
9,M.8668,Council Regulation 139/2004,2017-11-23,2017-11-24,1


In [23]:

# Stage 2.4 — Task 8
# Create case-level date quality indicators

import pandas as pd
import numpy as np

df_cases_clean = df_cases.copy()

# Parse original dates without overwriting them
df_cases_clean["notification_dt"] = pd.to_datetime(
    df_cases_clean["notification_date"],
    errors="coerce"
)

df_cases_clean["case_last_decision_dt"] = pd.to_datetime(
    df_cases_clean["last_recorded_decision_date"],
    errors="coerce"
)

# Identify latest decision recorded in the decision table
latest_dates = (
    df_decisions.assign(
        decision_dt=pd.to_datetime(
            df_decisions["decision_date"],
            errors="coerce"
        )
    )
    .groupby("case_id")["decision_dt"]
    .max()
    .rename("latest_nested_decision_dt")
)

df_cases_clean = df_cases_clean.merge(
    latest_dates,
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Record the difference between source date fields
df_cases_clean["date_difference_days"] = (
    df_cases_clean["latest_nested_decision_dt"]
    - df_cases_clean["case_last_decision_dt"]
).dt.days

# Create date-quality categories
df_cases_clean["date_quality_flag"] = np.select(
    [
        df_cases_clean["case_last_decision_dt"].isna(),
        df_cases_clean["latest_nested_decision_dt"].isna(),
        df_cases_clean["date_difference_days"].eq(0),
        df_cases_clean["date_difference_days"].eq(1)
    ],
    [
        "No case-level decision date",
        "No nested decision date",
        "Dates agree",
        "One-day discrepancy"
    ],
    default="Other discrepancy"
)

# Flag case-level chronological issues separately
df_cases_clean["chronology_flag"] = np.select(
    [
        df_cases_clean["notification_dt"].isna(),
        df_cases_clean["case_last_decision_dt"].isna(),
        (
            df_cases_clean["case_last_decision_dt"]
            < df_cases_clean["notification_dt"]
        )
    ],
    [
        "Notification missing",
        "Decision date missing",
        "Decision before notification"
    ],
    default="No negative chronology"
)

print("=== DATE QUALITY FLAGS ===")
print(df_cases_clean["date_quality_flag"].value_counts())

print("\n=== CHRONOLOGY FLAGS ===")
print(df_cases_clean["chronology_flag"].value_counts())

print("\n=== FINAL VALIDATION ===")
print("Total cases:", len(df_cases_clean))
print(
    "Duplicate case IDs:",
    df_cases_clean["case_id"].duplicated().sum()
)

print("\n=== EXCEPTIONAL CASES ===")
display(
    df_cases_clean[
        df_cases_clean["case_id"].isin(
            ["M.3333", "M.9142", "M.12355"]
        )
    ][[
        "case_id",
        "notification_date",
        "last_recorded_decision_date",
        "latest_nested_decision_dt",
        "date_difference_days",
        "date_quality_flag",
        "chronology_flag"
    ]]
)


=== DATE QUALITY FLAGS ===
date_quality_flag
Dates agree                    10266
One-day discrepancy               97
No case-level decision date       40
Other discrepancy                  1
Name: count, dtype: int64

=== CHRONOLOGY FLAGS ===
chronology_flag
No negative chronology          10099
Notification missing              273
Decision date missing              31
Decision before notification        1
Name: count, dtype: int64

=== FINAL VALIDATION ===
Total cases: 10404
Duplicate case IDs: 0

=== EXCEPTIONAL CASES ===


,case_id,notification_date,last_recorded_decision_date,latest_nested_decision_dt,date_difference_days,date_quality_flag,chronology_flag
691,M.3333,2007-01-31,2007-10-03,2007-10-03,0.0,Dates agree,No negative chronology
7714,M.9142,2019-07-15,2019-08-07,2019-07-08,-30.0,Other discrepancy,No negative chronology
10395,M.12355,2026-10-29,2026-05-07,2026-05-07,0.0,Dates agree,Decision before notification


In [24]:

# Stage 2.5 — Task 1
# Create provisional regulatory decision categories

import pandas as pd

df_decisions_clean = df_decisions.copy()

df_decisions_clean = df_decisions_clean.merge(
    df_cases[["case_id", "regulation"]],
    on="case_id",
    how="left",
    validate="many_to_one"
)

def classify_decision(row):
    label = row["decision_type_labels"]
    regulation = row["regulation"]

    if label == "Art. 6(1)(b)":
        return "Phase I - Unconditional clearance"

    if label == "Art. 6(1)(b) with conditions & obligations":
        return "Phase I - Conditional clearance"

    if label == "Art. 8(1)" and regulation == "Council Regulation 139/2004":
        return "Phase II - Unconditional clearance"

    if label == "Art. 8(2)" and regulation == "Council Regulation 4064/89":
        return "Phase II - Unconditional clearance"

    if label == "Art. 8(2) with conditions & obligations":
        return "Phase II - Conditional clearance"

    if label == "Art. 8(3)":
        return "Prohibition"

    if label == "Art. 6(1)(c)":
        return "Phase II initiated"

    if label == "Art. 6(1)(a)":
        return "Outside regulation scope"

    if label in {
        "Withdrawn",
        "Aborted / withdrawn",
        "Aborted / withdrawn (N/2)"
    }:
        return "Withdrawal / abandonment"

    if label in {"Art. 8(1)", "Art. 8(2)"}:
        return "Regulation classification review"

    return "Other procedural decision"

df_decisions_clean["decision_category"] = (
    df_decisions_clean.apply(classify_decision, axis=1)
)

print("=== DECISION CATEGORY DISTRIBUTION ===")
print(
    df_decisions_clean["decision_category"]
    .value_counts()
    .to_string()
)

print("\n=== CLASSIFICATION VALIDATION ===")
print("Original decision rows:", len(df_decisions))
print("Classified decision rows:", len(df_decisions_clean))
print(
    "Unclassified values:",
    df_decisions_clean["decision_category"].isna().sum()
)
print(
    "Regulation classification reviews:",
    (
        df_decisions_clean["decision_category"]
        == "Regulation classification review"
    ).sum()
)

print("\n=== OTHER PROCEDURAL DECISION TYPES ===")
print(
    df_decisions_clean.loc[
        df_decisions_clean["decision_category"]
        == "Other procedural decision",
        "decision_type_labels"
    ].value_counts().to_string()
)

assert len(df_decisions_clean) == len(df_decisions)
print("\nPASS: Original decision row count preserved")


=== DECISION CATEGORY DISTRIBUTION ===
decision_category
Phase I - Unconditional clearance     9176
Other procedural decision             1191
Phase I - Conditional clearance        371
Phase II initiated                     311
Withdrawal / abandonment               266
Phase II - Conditional clearance       154
Phase II - Unconditional clearance      67
Outside regulation scope                56
Prohibition                             33

=== CLASSIFICATION VALIDATION ===
Original decision rows: 11625
Classified decision rows: 11625
Unclassified values: 0
Regulation classification reviews: 0

=== OTHER PROCEDURAL DECISION TYPES ===
decision_type_labels
Art. 4(4) - Full referral                                     223
Art. 18 (Statement of Objections)                             166
Art. 4(5) - Referral                                          160
Purchaser approval                                            151
Art. 10(3)                                                    110
Art. 7(

In [25]:

# =====================================================
# Stage 2.5 — Task 1
# Create Provisional Regulatory Decision Categories
# =====================================================

import pandas as pd

# Preserve the original decision table
df_decisions_clean = df_decisions.copy()

# Attach the applicable EU merger regulation
df_decisions_clean = df_decisions_clean.merge(
    df_cases[["case_id", "regulation"]],
    on="case_id",
    how="left",
    validate="many_to_one"
)

# Define provisional regulatory categories
def classify_decision(row):

    label = row["decision_type_labels"]
    regulation = row["regulation"]

    # Phase I clearance
    if label == "Art. 6(1)(b)":
        return "Phase I - Unconditional clearance"

    if label == "Art. 6(1)(b) with conditions & obligations":
        return "Phase I - Conditional clearance"

    # Phase II unconditional clearance
    if (
        label == "Art. 8(1)"
        and regulation == "Council Regulation 139/2004"
    ):
        return "Phase II - Unconditional clearance"

    if (
        label == "Art. 8(2)"
        and regulation == "Council Regulation 4064/89"
    ):
        return "Phase II - Unconditional clearance"

    # Phase II conditional clearance
    if label == "Art. 8(2) with conditions & obligations":
        return "Phase II - Conditional clearance"

    # Prohibition
    if label == "Art. 8(3)":
        return "Prohibition"

    # Phase II investigation initiated
    if label == "Art. 6(1)(c)":
        return "Phase II initiated"

    # Outside scope
    if label == "Art. 6(1)(a)":
        return "Outside regulation scope"

    # Withdrawals and abandoned transactions
    if label in {
        "Withdrawn",
        "Aborted / withdrawn",
        "Aborted / withdrawn (N/2)"
    }:
        return "Withdrawal / abandonment"

    # Categories requiring legal classification review
    if label in {"Art. 8(1)", "Art. 8(2)"}:
        return "Regulation classification review"

    # All remaining procedural decisions
    return "Other procedural decision"


# Apply classifications
df_decisions_clean["decision_category"] = (
    df_decisions_clean.apply(classify_decision, axis=1)
)

# =====================================================
# VALIDATION
# =====================================================

print("=== DECISION CATEGORY DISTRIBUTION ===")

print(
    df_decisions_clean["decision_category"]
    .value_counts()
    .to_string()
)

print("\n=== CLASSIFICATION VALIDATION ===")

print("Original decision rows:", len(df_decisions))

print("Classified decision rows:", len(df_decisions_clean))

print(
    "Unclassified values:",
    df_decisions_clean["decision_category"].isna().sum()
)

print(
    "Regulation classification reviews:",
    (
        df_decisions_clean["decision_category"]
        == "Regulation classification review"
    ).sum()
)

print("\n=== OTHER PROCEDURAL DECISION TYPES ===")

print(
    df_decisions_clean.loc[
        df_decisions_clean["decision_category"]
        == "Other procedural decision",
        "decision_type_labels"
    ].value_counts().to_string()
)

# Confirm original row count is preserved
assert len(df_decisions_clean) == len(df_decisions)

print("\nPASS: Original decision row count preserved")


=== DECISION CATEGORY DISTRIBUTION ===
decision_category
Phase I - Unconditional clearance     9176
Other procedural decision             1191
Phase I - Conditional clearance        371
Phase II initiated                     311
Withdrawal / abandonment               266
Phase II - Conditional clearance       154
Phase II - Unconditional clearance      67
Outside regulation scope                56
Prohibition                             33

=== CLASSIFICATION VALIDATION ===
Original decision rows: 11625
Classified decision rows: 11625
Unclassified values: 0
Regulation classification reviews: 0

=== OTHER PROCEDURAL DECISION TYPES ===
decision_type_labels
Art. 4(4) - Full referral                                     223
Art. 18 (Statement of Objections)                             166
Art. 4(5) - Referral                                          160
Purchaser approval                                            151
Art. 10(3)                                                    110
Art. 7(

In [26]:

# Stage 2.5 — Task 2
# Audit multiple outcome-related decisions per case

import pandas as pd

# Categories potentially relevant to final outcomes
outcome_categories = [
    "Phase I - Unconditional clearance",
    "Phase I - Conditional clearance",
    "Phase II - Unconditional clearance",
    "Phase II - Conditional clearance",
    "Prohibition",
    "Withdrawal / abandonment",
    "Outside regulation scope"
]

# Retain outcome-related decision records
df_outcome_audit = df_decisions_clean[
    df_decisions_clean["decision_category"].isin(
        outcome_categories
    )
].copy()

# Count outcome-related decisions per case
outcome_counts = (
    df_outcome_audit.groupby("case_id")
    .size()
)

total_cases = len(df_cases)

print("=== OUTCOME DECISION AUDIT ===")

print("Total merger cases:", total_cases)

print(
    "Cases with no outcome-related decision:",
    total_cases - len(outcome_counts)
)

print(
    "Cases with exactly one outcome-related decision:",
    (outcome_counts == 1).sum()
)

print(
    "Cases with multiple outcome-related decisions:",
    (outcome_counts > 1).sum()
)

print(
    "Maximum outcome-related decisions in one case:",
    outcome_counts.max()
)

print("\n=== CASE COVERAGE BY OUTCOME CATEGORY ===")

print(
    df_outcome_audit.groupby("decision_category")[
        "case_id"
    ].nunique().sort_values(ascending=False)
    .to_string()
)

# Identify cases requiring further investigation
multiple_ids = outcome_counts[
    outcome_counts > 1
].index

print("\n=== SAMPLE MULTIPLE-OUTCOME CASES ===")

sample_multiple = df_outcome_audit[
    df_outcome_audit["case_id"].isin(multiple_ids)
].copy()

sample_multiple = sample_multiple.merge(
    df_cases[["case_id", "case_title"]],
    on="case_id",
    how="left",
    validate="many_to_one"
)

display(
    sample_multiple[
        [
            "case_id",
            "case_title",
            "decision_date",
            "decision_type_labels",
            "decision_category"
        ]
    ].sort_values(
        ["case_id", "decision_date"]
    ).head(20)
)

print("\n=== VALIDATION ===")

classified_cases = (
    (total_cases - len(outcome_counts))
    + (outcome_counts == 1).sum()
    + (outcome_counts > 1).sum()
)

print("Cases accounted for:", classified_cases)

assert classified_cases == total_cases
print("PASS: All merger cases accounted for")


=== OUTCOME DECISION AUDIT ===
Total merger cases: 10404
Cases with no outcome-related decision: 359
Cases with exactly one outcome-related decision: 9970
Cases with multiple outcome-related decisions: 75
Maximum outcome-related decisions in one case: 3

=== CASE COVERAGE BY OUTCOME CATEGORY ===
decision_category
Phase I - Unconditional clearance     9174
Phase I - Conditional clearance        367
Withdrawal / abandonment               264
Phase II - Conditional clearance       152
Phase II - Unconditional clearance      66
Outside regulation scope                56
Prohibition                             33

=== SAMPLE MULTIPLE-OUTCOME CASES ===


,case_id,case_title,decision_date,decision_type_labels,decision_category
119,M.10407,AIP / ALVANCE DUNKERQUE TARGET BUSINESS,2021-07-19,Withdrawn,Withdrawal / abandonment
120,M.10407,AIP / ALVANCE DUNKERQUE TARGET BUSINESS,2021-08-27,Withdrawn,Withdrawal / abandonment
121,M.10407,AIP / ALVANCE DUNKERQUE TARGET BUSINESS,2021-11-23,Art. 6(1)(b),Phase I - Unconditional clearance
135,M.10507,HITACHI RAIL / GROUND TRANSPORTATION SYSTEMS B...,2022-11-03,Withdrawn,Withdrawal / abandonment
134,M.10507,HITACHI RAIL / GROUND TRANSPORTATION SYSTEMS B...,2023-10-30,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
127,M.10560,SIKA / MBCC GROUP,2022-07-04,Withdrawn,Withdrawal / abandonment
126,M.10560,SIKA / MBCC GROUP,2023-02-08,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
131,M.10561,CINTRA / ABERTIS / ITINERE / BIP & DRIVE,2022-10-26,Withdrawn,Withdrawal / abandonment
130,M.10561,CINTRA / ABERTIS / ITINERE / BIP & DRIVE,2023-02-14,Art. 6(1)(b),Phase I - Unconditional clearance
125,M.10594,SECURITAS / STANLEY SECURITY,2022-04-11,Withdrawn,Withdrawal / abandonment



=== VALIDATION ===
Cases accounted for: 10404
PASS: All merger cases accounted for


In [27]:

# =====================================================
# Stage 2.5 — Task 3
# Analyse Multiple-Outcome Decision Sequences
# =====================================================

import pandas as pd
from collections import Counter

# Select the 75 cases with multiple outcome records
multiple_ids = outcome_counts[
    outcome_counts > 1
].index

multi = df_outcome_audit[
    df_outcome_audit["case_id"].isin(multiple_ids)
].copy()

# Sort decisions chronologically
multi["decision_dt"] = pd.to_datetime(
    multi["decision_date"],
    errors="coerce"
)

multi = multi.sort_values(
    ["case_id", "decision_dt", "decision_id"]
)

# Build a chronological category sequence per case
sequences = multi.groupby("case_id")[
    "decision_category"
].agg(list)

print("=== MULTIPLE-OUTCOME SUMMARY ===")

print("Cases investigated:", len(sequences))

print(
    "Cases with different outcome categories:",
    sequences.apply(
        lambda x: len(set(x)) > 1
    ).sum()
)

print(
    "Cases repeating the same outcome category:",
    sequences.apply(
        lambda x: len(set(x)) == 1
    ).sum()
)

# Identify withdrawal followed by clearance
def withdrawal_then_clearance(seq):
    for i, category in enumerate(seq):
        if category == "Withdrawal / abandonment":
            if any(
                "clearance" in later.lower()
                for later in seq[i + 1:]
            ):
                return True
    return False

print(
    "Cases with withdrawal followed by clearance:",
    sequences.apply(withdrawal_then_clearance).sum()
)

# Count chronological sequences
patterns = Counter(
    " -> ".join(seq)
    for seq in sequences
)

print("\n=== MOST COMMON DECISION SEQUENCES ===")

for pattern, count in patterns.most_common(15):
    print(f"{count} cases: {pattern}")

# Examine the last recorded outcome-related decision
last_decisions = multi.groupby("case_id").tail(1)

print("\n=== LAST RECORDED OUTCOME CATEGORIES ===")

print(
    last_decisions["decision_category"]
    .value_counts()
    .to_string()
)

# Identify same-date events that may need review
same_date_cases = multi.loc[
    multi.duplicated(
        ["case_id", "decision_dt"],
        keep=False
    ),
    "case_id"
].nunique()

print("\n=== ADDITIONAL VALIDATION ===")
print("Cases with same-date outcome records:", same_date_cases)

assert len(sequences) == 75
print("PASS: All 75 cases accounted for")

print("\n=== SAMPLE DECISION HISTORIES ===")

sample_ids = [
    "M.10407",
    "M.10507",
    "M.10560",
    "M.10658"
]

display(
    multi[multi["case_id"].isin(sample_ids)][
        [
            "case_id",
            "decision_date",
            "decision_type_labels",
            "decision_category"
        ]
    ]
)


=== MULTIPLE-OUTCOME SUMMARY ===
Cases investigated: 75
Cases with different outcome categories: 66
Cases repeating the same outcome category: 9
Cases with withdrawal followed by clearance: 53

=== MOST COMMON DECISION SEQUENCES ===
33 cases: Withdrawal / abandonment -> Phase I - Unconditional clearance
15 cases: Withdrawal / abandonment -> Phase I - Conditional clearance
5 cases: Outside regulation scope -> Phase I - Unconditional clearance
4 cases: Phase I - Conditional clearance -> Phase I - Conditional clearance
2 cases: Withdrawal / abandonment -> Withdrawal / abandonment -> Phase I - Unconditional clearance
2 cases: Withdrawal / abandonment -> Phase II - Unconditional clearance
2 cases: Phase II - Conditional clearance -> Phase II - Conditional clearance
2 cases: Phase I - Unconditional clearance -> Phase I - Unconditional clearance
1 cases: Phase I - Unconditional clearance -> Phase I - Conditional clearance
1 cases: Phase I - Conditional clearance -> Phase I - Unconditional cle

,case_id,decision_date,decision_type_labels,decision_category
9384,M.10407,2021-07-19,Withdrawn,Withdrawal / abandonment
9385,M.10407,2021-08-27,Withdrawn,Withdrawal / abandonment
9386,M.10407,2021-11-23,Art. 6(1)(b),Phase I - Unconditional clearance
10050,M.10507,2022-11-03,Withdrawn,Withdrawal / abandonment
10049,M.10507,2023-10-30,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
9686,M.10560,2022-07-04,Withdrawn,Withdrawal / abandonment
9685,M.10560,2023-02-08,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
10790,M.10658,2022-06-07,Withdrawn,Withdrawal / abandonment
10792,M.10658,2023-05-04,Art. 8(1),Phase II - Unconditional clearance


In [28]:

# =====================================================
# Stage 2.5 — Task 4
# Investigate Unusual Regulatory Decision Sequences
# =====================================================

from collections import Counter
import pandas as pd

review_records = []

# Use the 75 cases from Task 3
for case_id, group in multi.groupby("case_id"):

    group = group.sort_values(
        ["decision_dt", "decision_id"]
    )

    categories = group["decision_category"].tolist()
    flags = []

    # Check decisions with identical dates
    if group["decision_dt"].duplicated().any():
        flags.append("Same-date outcomes")

    # Earlier outside-scope decision followed by clearance
    if any(
        category == "Outside regulation scope"
        and any(
            "clearance" in later.lower()
            for later in categories[i + 1:]
        )
        for i, category in enumerate(categories)
    ):
        flags.append("Outside scope followed by clearance")

    # Earlier Phase II clearance followed by Phase I clearance
    if any(
        category.startswith("Phase II")
        and "clearance" in category.lower()
        and any(
            later.startswith("Phase I")
            and "clearance" in later.lower()
            for later in categories[i + 1:]
        )
        for i, category in enumerate(categories)
    ):
        flags.append("Phase II clearance followed by Phase I")

    # Earlier prohibition followed by clearance
    if any(
        category == "Prohibition"
        and any(
            "clearance" in later.lower()
            for later in categories[i + 1:]
        )
        for i, category in enumerate(categories)
    ):
        flags.append("Prohibition followed by clearance")

    # Different Phase I clearance classifications
    if (
        "Phase I - Unconditional clearance" in categories
        and "Phase I - Conditional clearance" in categories
    ):
        flags.append("Different Phase I clearance types")

    if flags:
        review_records.append({
            "case_id": case_id,
            "flags": flags,
            "sequence": " -> ".join(categories)
        })

review_cases = pd.DataFrame(
    review_records,
    columns=["case_id", "flags", "sequence"]
)

print("=== EXCEPTION REVIEW SUMMARY ===")
print("Multiple-outcome cases assessed:", 75)
print("Cases flagged for further review:", len(review_cases))

flag_counts = Counter(
    flag
    for flags in review_cases["flags"]
    for flag in flags
)

print("\n=== EXCEPTION TYPES ===")
for flag, count in flag_counts.most_common():
    print(f"{flag}: {count}")

print("\n=== FLAGGED CASE HISTORIES ===")

for _, row in review_cases.iterrows():
    case_id = row["case_id"]

    print(f"\nCase: {case_id}")
    print("Flags:", ", ".join(row["flags"]))

    history = multi[multi["case_id"] == case_id]

    for _, decision in history.iterrows():
        print(
            f"  {decision['decision_date']} | "
            f"{decision['decision_category']}"
        )

print("\n=== VALIDATION ===")
print(
    "Unique flagged cases:",
    review_cases["case_id"].nunique()
)
print("Review completed")


=== EXCEPTION REVIEW SUMMARY ===
Multiple-outcome cases assessed: 75
Cases flagged for further review: 16

=== EXCEPTION TYPES ===
Phase II clearance followed by Phase I: 6
Same-date outcomes: 5
Outside scope followed by clearance: 5
Different Phase I clearance types: 2
Prohibition followed by clearance: 2

=== FLAGGED CASE HISTORIES ===

Case: M.1397
Flags: Different Phase I clearance types
  1999-03-15 | Phase I - Unconditional clearance
  1999-05-17 | Phase I - Conditional clearance

Case: M.1587
Flags: Same-date outcomes, Outside scope followed by clearance
  1999-11-04 | Outside regulation scope
  1999-11-04 | Phase I - Unconditional clearance

Case: M.1795
Flags: Different Phase I clearance types
  2000-04-12 | Phase I - Conditional clearance
  2002-07-26 | Phase I - Unconditional clearance

Case: M.197
Flags: Same-date outcomes, Outside scope followed by clearance
  1992-04-30 | Outside regulation scope
  1992-04-30 | Phase I - Unconditional clearance

Case: M.2416
Flags: Prohib

In [29]:

# Stage 2.5 — Task 5
# Correct unusual decision-sequence classifications

import pandas as pd
from collections import Counter

phase1 = {
    "Phase I - Unconditional clearance",
    "Phase I - Conditional clearance"
}

phase2 = {
    "Phase II - Unconditional clearance",
    "Phase II - Conditional clearance"
}

clearances = phase1 | phase2
review_records_corrected = []

for case_id, group in multi.groupby("case_id"):

    g = group.sort_values(
        ["decision_dt", "decision_id"]
    )

    events = list(zip(
        g["decision_dt"],
        g["decision_category"]
    ))

    flags = []

    if g["decision_dt"].duplicated().any():
        flags.append("Same-date outcomes")

    if any(
        c1 == "Outside regulation scope"
        and c2 in clearances
        and d1 < d2
        for d1, c1 in events
        for d2, c2 in events
    ):
        flags.append("Outside scope then later clearance")

    if any(
        c1 == "Outside regulation scope"
        and c2 in clearances
        and d1 == d2
        for d1, c1 in events
        for d2, c2 in events
    ):
        flags.append("Outside scope and clearance on same date")

    if any(
        c1 in phase2
        and c2 in phase1
        and d1 < d2
        for d1, c1 in events
        for d2, c2 in events
    ):
        flags.append("Phase II then later Phase I clearance")

    if any(
        c1 == "Prohibition"
        and c2 in clearances
        and d1 < d2
        for d1, c1 in events
        for d2, c2 in events
    ):
        flags.append("Prohibition then later clearance")

    if phase1.issubset(set(g["decision_category"])):
        flags.append("Different Phase I clearance types")

    if flags:
        review_records_corrected.append({
            "case_id": case_id,
            "flags": flags
        })

review_corrected = pd.DataFrame(
    review_records_corrected
)

print("=== CORRECTED EXCEPTION SUMMARY ===")
print("Multiple-outcome cases assessed:", multi["case_id"].nunique())
print("Cases requiring review:", len(review_corrected))

print("\n=== CORRECTED EXCEPTION TYPES ===")

counts = Counter(
    flag
    for flags in review_corrected["flags"]
    for flag in flags
)

for flag, count in counts.most_common():
    print(f"{flag}: {count}")

print("\n=== FLAGGED CASES ===")

for _, row in review_corrected.iterrows():
    print(
        row["case_id"],
        "|",
        ", ".join(row["flags"])
    )

print("\n=== VALIDATION ===")
assert multi["case_id"].nunique() == 75
print("PASS: All 75 multiple-outcome cases assessed")


=== CORRECTED EXCEPTION SUMMARY ===
Multiple-outcome cases assessed: 75
Cases requiring review: 12

=== CORRECTED EXCEPTION TYPES ===
Same-date outcomes: 5
Outside scope and clearance on same date: 5
Different Phase I clearance types: 2
Prohibition then later clearance: 2
Phase II then later Phase I clearance: 2
Outside scope then later clearance: 1

=== FLAGGED CASES ===
M.1397 | Different Phase I clearance types
M.1587 | Same-date outcomes, Outside scope and clearance on same date
M.1795 | Different Phase I clearance types
M.197 | Same-date outcomes, Outside scope and clearance on same date
M.2416 | Prohibition then later clearance
M.308 | Phase II then later Phase I clearance
M.4 | Same-date outcomes, Outside scope and clearance on same date
M.409 | Same-date outcomes, Outside scope and clearance on same date
M.479 | Same-date outcomes, Outside scope and clearance on same date
M.553 | Prohibition then later clearance
M.7000 | Phase II then later Phase I clearance
M.7538 | Outside sc

In [30]:

# =====================================================
# Stage 2.5 — Task 6
# Investigate Cases Without Outcome-Related Decisions
# =====================================================

import pandas as pd

# Identify all cases with outcome-related decisions
cases_with_outcomes = set(
    df_outcome_audit["case_id"]
)

# Identify cases without such decisions
no_outcome_ids = (
    set(df_cases["case_id"])
    - cases_with_outcomes
)

# Retrieve their decision records
df_no_outcome = df_decisions_clean[
    df_decisions_clean["case_id"].isin(
        no_outcome_ids
    )
].copy()

print("=== NO-OUTCOME CASE SUMMARY ===")

print(
    "Cases without outcome-related decisions:",
    len(no_outcome_ids)
)

print(
    "Cases with no decision records:",
    sum(
        df_cases.set_index("case_id")
        .loc[list(no_outcome_ids), "decision_count"]
        == 0
    )
)

print(
    "Cases with procedural decisions only:",
    df_no_outcome["case_id"].nunique()
)

print("\n=== PROCEDURAL DECISION TYPES ===")

print(
    df_no_outcome["decision_type_labels"]
    .value_counts()
    .to_string()
)

print("\n=== EXAMPLE CASES ===")

examples = (
    df_cases[
        df_cases["case_id"].isin(no_outcome_ids)
    ][
        ["case_id", "case_title",
         "decision_count", "regulation"]
    ]
    .sort_values("case_id")
    .head(15)
)

display(examples)

print("\n=== VALIDATION ===")

print(
    "Cases with outcome-related decisions:",
    len(cases_with_outcomes)
)

print(
    "Cases without outcome-related decisions:",
    len(no_outcome_ids)
)

assert (
    len(cases_with_outcomes)
    + len(no_outcome_ids)
    == len(df_cases)
)

print("PASS: All 10,404 cases accounted for")


=== NO-OUTCOME CASE SUMMARY ===
Cases without outcome-related decisions: 359
Cases with no decision records: 40
Cases with procedural decisions only: 319

=== PROCEDURAL DECISION TYPES ===
decision_type_labels
Art. 4(4) - Full referral              220
Art. 9(3) full referral                 51
Art. 18 (Statement of Objections)       18
Art. 14                                 15
Art. 7(3)                                7
Art. 9(3) partial referral               6
Art. 22 Refusal of referral              4
Art. 21(4)                               3
Art. 6(1)(c)                             3
Art. 21(3)                               2
Art. 4(4) - Partial referral             2
Rejection of a request to act            2
2/3 rule                                 1
Closure of proceedings                   1
Withdrawal of Art. 8(5)(c) decision      1
Art. 8(5)(c)                             1
Art. 10(3)                               1
Withdrawal of Art. 14 decision           1
Art. 21(4) Preli

,case_id,case_title,decision_count,regulation
2718,C.1886,ILIAD,1,Council Regulation 139/2004
2720,C.1887,MEDIASET,1,Council Regulation 139/2004
7888,M.10015,CARREFOUR / SUPERSOL,1,Council Regulation 139/2004
7886,M.10016,BELLIS / ASDA,1,Council Regulation 139/2004
8494,M.10126,APOLLO MANAGEMENT / LOTTOMATICA VIDEOLOT RETE ...,1,Council Regulation 139/2004
8529,M.10134,EG GROUP / OMV GERMANY BUSINESS,1,Council Regulation 139/2004
8789,M.10144,K+S MINERALS AND AGRICULTURE / REMEX / JV,1,Council Regulation 139/2004
8787,M.10146,PENTA INVESTMENTS / ASSECO POLAND / SUPERNIUS ...,1,Council Regulation 139/2004
8686,M.10186,ELSAN / C2S,1,Council Regulation 139/2004
9237,M.10201,AHOLD DELHAIZE / DEEN ASSETS,1,Council Regulation 139/2004



=== VALIDATION ===
Cases with outcome-related decisions: 10045
Cases without outcome-related decisions: 359
PASS: All 10,404 cases accounted for


In [31]:

# =====================================================
# Stage 2.5 — Task 7
# Classify Cases Without Selected Outcomes
# =====================================================

import pandas as pd
from collections import Counter

referral_labels = {
    "Art. 4(4) - Full referral",
    "Art. 4(4) - Partial referral",
    "Art. 4(5) - Referral",
    "Art. 9(3) full referral",
    "Art. 9(3) partial referral",
    "Art. 22 Full referral"
}

case_groups = []

for case_id in sorted(no_outcome_ids):

    decisions = df_no_outcome.loc[
        df_no_outcome["case_id"] == case_id
    ]

    labels = set(
        decisions["decision_type_labels"]
        .dropna()
    )

    if len(decisions) == 0:
        category = "No recorded decisions"

    elif "Art. 6(1)(c)" in labels:
        category = "Phase II initiated - no selected outcome"

    elif labels.intersection(referral_labels):
        category = "Referral recorded - no selected outcome"

    else:
        category = "Other procedural decisions only"

    case_groups.append({
        "case_id": case_id,
        "category": category,
        "decision_count": len(decisions)
    })

df_no_outcome_groups = pd.DataFrame(case_groups)

print("=== EXCLUSIVE CASE CATEGORIES ===")

print(
    df_no_outcome_groups["category"]
    .value_counts()
    .to_string()
)

print("\n=== VALIDATION ===")

print("Cases classified:", len(df_no_outcome_groups))

print(
    "Duplicate case IDs:",
    df_no_outcome_groups["case_id"].duplicated().sum()
)

assert len(df_no_outcome_groups) == 359
assert df_no_outcome_groups["case_id"].is_unique

print("PASS: All 359 cases classified exactly once")

print("\n=== PHASE II CASES WITHOUT SELECTED OUTCOME ===")

print(
    df_no_outcome_groups.loc[
        df_no_outcome_groups["category"]
        == "Phase II initiated - no selected outcome",
        "case_id"
    ].tolist()
)

print("\n=== NON-M CASE IDENTIFIERS ===")

non_m_cases = df_cases[
    ~df_cases["case_id"].str.startswith("M.")
]

print("Non-M case count:", len(non_m_cases))

display(
    non_m_cases[
        ["case_id", "case_title", "regulation",
         "decision_count"]
    ].head(15)
)

print("\n=== SAMPLE OTHER PROCEDURAL CASES ===")

display(
    df_no_outcome_groups[
        df_no_outcome_groups["category"]
        == "Other procedural decisions only"
    ].head(10)
)


=== EXCLUSIVE CASE CATEGORIES ===
category
Referral recorded - no selected outcome     278
No recorded decisions                        40
Other procedural decisions only              38
Phase II initiated - no selected outcome      3

=== VALIDATION ===
Cases classified: 359
Duplicate case IDs: 0
PASS: All 359 cases classified exactly once

=== PHASE II CASES WITHOUT SELECTED OUTCOME ===
['M.11895', 'M.12236', 'M.12270']

=== NON-M CASE IDENTIFIERS ===
Non-M case count: 2


,case_id,case_title,regulation,decision_count
2718,C.1886,ILIAD,Council Regulation 139/2004,1
2720,C.1887,MEDIASET,Council Regulation 139/2004,1



=== SAMPLE OTHER PROCEDURAL CASES ===


,case_id,category,decision_count
0,C.1886,Other procedural decisions only,1
1,C.1887,Other procedural decisions only,1
18,M.10483.AP,Other procedural decisions only,3
19,M.10493.AP,Other procedural decisions only,3
20,M.10494.AP,Other procedural decisions only,1
31,M.10938.AP,Other procedural decisions only,2
32,M.10939.AP,Other procedural decisions only,4
33,M.10962.AP,Other procedural decisions only,2
39,M.11184.AP,Other procedural decisions only,2
69,M.12052.AP,Other procedural decisions only,1


In [32]:

# =====================================================
# Stage 2.5 — Task 8
# Audit Non-Standard Merger Case Identifiers
# =====================================================

import pandas as pd
import re
import json

# Preserve original case records
df_identifier_audit = df_cases.copy()

# Classify identifier patterns
def classify_identifier(case_id):

    if re.fullmatch(r"M\.\d+", case_id):
        return "Standard M identifier"

    elif re.fullmatch(r"M\.\d+\.AP", case_id):
        return "M identifier with AP suffix"

    elif case_id.startswith("M."):
        return "Other M identifier variant"

    else:
        return "Non-M identifier"

df_identifier_audit["identifier_pattern"] = (
    df_identifier_audit["case_id"]
    .apply(classify_identifier)
)

print("=== IDENTIFIER PATTERN DISTRIBUTION ===")

print(
    df_identifier_audit["identifier_pattern"]
    .value_counts()
    .to_string()
)

# Identify cases outside the standard format
nonstandard = df_identifier_audit[
    df_identifier_audit["identifier_pattern"]
    != "Standard M identifier"
].copy()

# Inspect available case metadata
def get_metadata_value(case_id, field):

    values = data[case_id].get(
        "metadata", {}
    ).get(field, [])

    if not values:
        return None

    value = values[0]

    if isinstance(value, str):
        try:
            decoded = json.loads(value)

            if isinstance(decoded, dict):
                return decoded.get("label", value)

        except (ValueError, TypeError):
            pass

    return value

nonstandard["case_type"] = (
    nonstandard["case_id"].apply(
        lambda x: get_metadata_value(x, "caseType")
    )
)

nonstandard["case_instrument"] = (
    nonstandard["case_id"].apply(
        lambda x: get_metadata_value(x, "caseInstrument")
    )
)

print("\n=== NON-STANDARD CASE TYPES ===")

print(
    nonstandard.groupby(
        ["identifier_pattern", "case_type"],
        dropna=False
    ).size().to_string()
)

print("\n=== NON-STANDARD CASE EXAMPLES ===")

display(
    nonstandard[[
        "case_id",
        "case_title",
        "identifier_pattern",
        "case_type",
        "case_instrument",
        "decision_count"
    ]].head(20)
)

print("\n=== VALIDATION ===")

print("Total cases:", len(df_identifier_audit))

print(
    "Standard identifiers:",
    (
        df_identifier_audit["identifier_pattern"]
        == "Standard M identifier"
    ).sum()
)

print("Non-standard identifiers:", len(nonstandard))

assert len(df_identifier_audit) == 10404

assert (
    df_identifier_audit["case_id"].is_unique
)

print("PASS: All case identifiers accounted for")


=== IDENTIFIER PATTERN DISTRIBUTION ===
identifier_pattern
Standard M identifier          10376
M identifier with AP suffix       26
Non-M identifier                   2

=== NON-STANDARD CASE TYPES ===
identifier_pattern           case_type           
M identifier with AP suffix  Additional Procedure    26
Non-M identifier             Consultation             2

=== NON-STANDARD CASE EXAMPLES ===


,case_id,case_title,identifier_pattern,case_type,case_instrument,decision_count
150,M.1610.AP,DEUTSCHE POST / TRANS-O-FLEX (Art.14 proc.),M identifier with AP suffix,Additional Procedure,Merger,1
182,M.1634.AP,MITSUBISHI HEAVY INDUSTRIES (Art.11 and Art. 1...,M identifier with AP suffix,Additional Procedure,Merger,1
229,M.2624.AP,BP / ERDÖLCHEMIE (Art. 14 proc.),M identifier with AP suffix,Additional Procedure,Merger,1
1028,M.1635.AP,BABCOCK & WILCOX (Art.11 proc.) (see M.1431),M identifier with AP suffix,Additional Procedure,Merger,1
1374,M.1724.AP,BSCH / CHAMPALIMAUD (Art.21 and Art.22 proc.) ...,M identifier with AP suffix,Additional Procedure,Merger,2
2331,M.1608.AP,KLM / MARTINAIR (Art.14.1 proc.),M identifier with AP suffix,Additional Procedure,Merger,1
2506,M.1543.AP,SANOFI / SYNTHELABO (Art.14 proc.) (see M.139...,M identifier with AP suffix,Additional Procedure,Merger,1
2718,C.1886,ILIAD,Non-M identifier,Consultation,Merger,1
2720,C.1887,MEDIASET,Non-M identifier,Consultation,Merger,1
2924,M.3255.AP,TETRA LAVAL / SIDEL (Art. 14.1 proc.),M identifier with AP suffix,Additional Procedure,Merger,1



=== VALIDATION ===
Total cases: 10404
Standard identifiers: 10376
Non-standard identifiers: 28
PASS: All case identifiers accounted for


In [33]:

# =====================================================
# Stage 2.5 — Task 9
# Verify Core Merger Case Population
# =====================================================

import pandas as pd
import json

df_scope_audit = df_identifier_audit.copy()

# Read the official case-type label
def extract_case_type(case_id):

    metadata = data[case_id].get("metadata", {})
    values = metadata.get("caseType", [])

    if not values:
        return None

    value = values[0]

    if isinstance(value, str):
        try:
            value = json.loads(value)
        except (ValueError, TypeError):
            return value

    if isinstance(value, dict):
        return value.get("label")

    return str(value)

df_scope_audit["case_type_label"] = (
    df_scope_audit["case_id"].apply(extract_case_type)
)

print("=== ALL CASE TYPES ===")
print(
    df_scope_audit["case_type_label"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n=== IDENTIFIER AND CASE TYPE CROSS-CHECK ===")

print(
    pd.crosstab(
        df_scope_audit["identifier_pattern"],
        df_scope_audit["case_type_label"],
        dropna=False
    ).to_string()
)

# Identify the proposed core population
core_mask = (
    df_scope_audit["identifier_pattern"].eq(
        "Standard M identifier"
    )
    & df_scope_audit["case_type_label"].eq("Merger")
)

df_core_candidate = df_scope_audit.loc[
    core_mask
].copy()

df_supplementary = df_scope_audit.loc[
    ~core_mask
].copy()

print("\n=== PROPOSED ANALYTICAL POPULATION ===")

print("Core merger candidates:", len(df_core_candidate))
print("Supplementary records:", len(df_supplementary))

print(
    "Missing case-type labels:",
    df_scope_audit["case_type_label"].isna().sum()
)

print("\n=== VALIDATION ===")

print("Total original records:", len(df_cases))

print(
    "Total classified records:",
    len(df_core_candidate) + len(df_supplementary)
)

assert (
    len(df_core_candidate) + len(df_supplementary)
    == len(df_cases)
)

assert df_scope_audit["case_id"].is_unique

print("PASS: All case records accounted for")


=== ALL CASE TYPES ===
case_type_label
Merger                  10376
Additional Procedure       26
Consultation                2

=== IDENTIFIER AND CASE TYPE CROSS-CHECK ===
case_type_label              Additional Procedure  Consultation  Merger
identifier_pattern                                                     
M identifier with AP suffix                    26             0       0
Non-M identifier                                0             2       0
Standard M identifier                           0             0   10376

=== PROPOSED ANALYTICAL POPULATION ===
Core merger candidates: 10376
Supplementary records: 28
Missing case-type labels: 0

=== VALIDATION ===
Total original records: 10404
Total classified records: 10404
PASS: All case records accounted for


In [34]:

# =====================================================
# Stage 2.5 — Task 10
# Create Core EU Merger Analytical Tables
# =====================================================

# Identify validated core merger case IDs
core_ids = set(df_core_candidate["case_id"])

# 1. Create core case-level table
df_cases_core = df_cases_clean[
    df_cases_clean["case_id"].isin(core_ids)
].copy()

# 2. Create core decision-level table
df_decisions_core = df_decisions_clean[
    df_decisions_clean["case_id"].isin(core_ids)
].copy()

# 3. Create core sector-level table
df_sectors_core = df_sectors[
    df_sectors["case_id"].isin(core_ids)
].copy()

# 4. Preserve supplementary records separately
df_cases_supplementary = df_cases_clean[
    ~df_cases_clean["case_id"].isin(core_ids)
].copy()

df_decisions_supplementary = df_decisions_clean[
    ~df_decisions_clean["case_id"].isin(core_ids)
].copy()

df_sectors_supplementary = df_sectors[
    ~df_sectors["case_id"].isin(core_ids)
].copy()

print("=== CORE ANALYTICAL TABLES ===")

print("Core merger cases:", len(df_cases_core))
print("Core decision records:", len(df_decisions_core))
print("Core sector records:", len(df_sectors_core))

print("\n=== SUPPLEMENTARY TABLES ===")

print("Supplementary cases:", len(df_cases_supplementary))
print("Supplementary decisions:", len(df_decisions_supplementary))
print("Supplementary sectors:", len(df_sectors_supplementary))

print("\n=== DATA RECONCILIATION ===")

print(
    "Case records reconciled:",
    len(df_cases_core) + len(df_cases_supplementary)
)

print(
    "Decision records reconciled:",
    len(df_decisions_core)
    + len(df_decisions_supplementary)
)

print(
    "Sector records reconciled:",
    len(df_sectors_core)
    + len(df_sectors_supplementary)
)

print("\n=== CORE INTEGRITY CHECKS ===")

print(
    "Duplicate core case IDs:",
    df_cases_core["case_id"].duplicated().sum()
)

print(
    "Orphan core decisions:",
    len(
        set(df_decisions_core["case_id"])
        - set(df_cases_core["case_id"])
    )
)

print(
    "Orphan core sectors:",
    len(
        set(df_sectors_core["case_id"])
        - set(df_cases_core["case_id"])
    )
)

assert len(df_cases_core) == 10376

assert len(df_cases_supplementary) == 28

assert (
    len(df_cases_core) + len(df_cases_supplementary)
    == 10404
)

assert (
    len(df_decisions_core)
    + len(df_decisions_supplementary)
    == 11625
)

assert (
    len(df_sectors_core)
    + len(df_sectors_supplementary)
    == 16310
)

assert df_cases_core["case_id"].is_unique

assert set(df_decisions_core["case_id"]).issubset(core_ids)
assert set(df_sectors_core["case_id"]).issubset(core_ids)

print("\nPASS: Core tables created and reconciled")


=== CORE ANALYTICAL TABLES ===
Core merger cases: 10376
Core decision records: 11579
Core sector records: 16286

=== SUPPLEMENTARY TABLES ===
Supplementary cases: 28
Supplementary decisions: 46
Supplementary sectors: 24

=== DATA RECONCILIATION ===
Case records reconciled: 10404
Decision records reconciled: 11625
Sector records reconciled: 16310

=== CORE INTEGRITY CHECKS ===
Duplicate core case IDs: 0
Orphan core decisions: 0
Orphan core sectors: 0

PASS: Core tables created and reconciled


In [35]:

# =====================================================
# Stage 2.5 — Task 11
# Core Merger Case Outcome Assessment
# =====================================================

import pandas as pd
import numpy as np

# Select outcome-related decisions from core cases
core_outcomes = df_decisions_core[
    df_decisions_core["decision_category"].isin(
        outcome_categories
    )
].copy()

# Convert dates and order decisions chronologically
core_outcomes["decision_dt"] = pd.to_datetime(
    core_outcomes["decision_date"],
    errors="coerce"
)

core_outcomes = core_outcomes.sort_values(
    ["case_id", "decision_dt", "decision_id"]
)

# Count outcome-related records per case
outcome_counts_core = (
    core_outcomes.groupby("case_id")
    .size()
    .rename("outcome_record_count")
)

# Extract latest recorded outcome-related decision
latest_outcomes = (
    core_outcomes.groupby("case_id")
    .tail(1)[
        ["case_id", "decision_date", "decision_category"]
    ]
    .rename(columns={
        "decision_date": "latest_outcome_date",
        "decision_category": "latest_outcome_category"
    })
)

# Create one record per core merger case
df_case_outcome_assessment = (
    df_cases_core[["case_id", "case_title"]]
    .merge(
        outcome_counts_core,
        on="case_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        latest_outcomes,
        on="case_id",
        how="left",
        validate="one_to_one"
    )
)

df_case_outcome_assessment[
    "outcome_record_count"
] = (
    df_case_outcome_assessment["outcome_record_count"]
    .fillna(0)
    .astype(int)
)

# Preserve the 12 previously flagged exception cases
flagged_ids = set(review_corrected["case_id"])

df_case_outcome_assessment["assessment_status"] = np.select(
    [
        df_case_outcome_assessment["outcome_record_count"].eq(0),
        df_case_outcome_assessment["case_id"].isin(flagged_ids),
        df_case_outcome_assessment["outcome_record_count"].gt(1)
    ],
    [
        "No selected outcome record",
        "Special sequence review required",
        "Multiple outcome records"
    ],
    default="Single outcome-related record"
)

print("=== CORE CASE OUTCOME ASSESSMENT ===")

print(
    df_case_outcome_assessment["assessment_status"]
    .value_counts()
    .to_string()
)

print("\n=== LATEST RECORDED OUTCOME CATEGORIES ===")

print(
    df_case_outcome_assessment["latest_outcome_category"]
    .fillna("No selected outcome")
    .value_counts()
    .to_string()
)

print("\n=== VALIDATION ===")

print("Core merger cases:", len(df_case_outcome_assessment))

print(
    "Duplicate case IDs:",
    df_case_outcome_assessment["case_id"]
    .duplicated()
    .sum()
)

print(
    "Cases requiring special sequence review:",
    (
        df_case_outcome_assessment["assessment_status"]
        == "Special sequence review required"
    ).sum()
)

assert len(df_case_outcome_assessment) == 10376
assert df_case_outcome_assessment["case_id"].is_unique

print("PASS: All core merger cases accounted for")

print("\n=== SAMPLE CASE ASSESSMENTS ===")

display(
    df_case_outcome_assessment[
        [
            "case_id",
            "case_title",
            "outcome_record_count",
            "latest_outcome_category",
            "assessment_status"
        ]
    ].head(10)
)


=== CORE CASE OUTCOME ASSESSMENT ===
assessment_status
Single outcome-related record       9969
No selected outcome record           332
Multiple outcome records              63
Special sequence review required      12

=== LATEST RECORDED OUTCOME CATEGORIES ===
latest_outcome_category
Phase I - Unconditional clearance     9172
Phase I - Conditional clearance        365
No selected outcome                    332
Withdrawal / abandonment               210
Phase II - Conditional clearance       149
Phase II - Unconditional clearance      66
Outside regulation scope                51
Prohibition                             31

=== VALIDATION ===
Core merger cases: 10376
Duplicate case IDs: 0
Cases requiring special sequence review: 12
PASS: All core merger cases accounted for

=== SAMPLE CASE ASSESSMENTS ===


,case_id,case_title,outcome_record_count,latest_outcome_category,assessment_status
0,M.3359,ANGLO AMERICAN / BAUERNFEIND,1,Phase I - Unconditional clearance,Single outcome-related record
1,M.2027,DEUTSCHE BANK / SAP / JV,1,Phase I - Unconditional clearance,Single outcome-related record
2,M.3358,HOGG ROBINSON / KUONI,1,Phase I - Unconditional clearance,Single outcome-related record
3,M.2028,ABB / BILFINGER / MVV ENERGIE / JV,1,Phase I - Unconditional clearance,Single outcome-related record
4,M.422,UNILEVER FRANCE / ORTIZ MIKO (II),1,Phase I - Conditional clearance,Single outcome-related record
5,M.2029,TATE & LYLE / AMYLUM,1,Phase I - Unconditional clearance,Single outcome-related record
6,M.420,KPR / CGP,1,Phase I - Unconditional clearance,Single outcome-related record
7,M.3355,APOLLO / JP MORGAN / PRIMACOM,1,Phase I - Unconditional clearance,Single outcome-related record
8,M.2023,BRAMBLES / ERMEWA / JV,1,Phase I - Unconditional clearance,Single outcome-related record
9,M.425,BS / BT,1,Outside regulation scope,Single outcome-related record


In [36]:

# =====================================================
# Stage 2.5 — Task 12
# Investigate Core Cases Without Selected Outcomes
# =====================================================

import pandas as pd

# Select core cases without a provisional outcome
core_no_outcome = df_case_outcome_assessment.loc[
    df_case_outcome_assessment["assessment_status"]
    == "No selected outcome record",
    ["case_id", "case_title"]
].copy()

# Attach the procedural categories previously established
core_no_outcome = core_no_outcome.merge(
    df_no_outcome_groups[
        ["case_id", "category", "decision_count"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one"
)

print("=== CORE NO-OUTCOME CLASSIFICATIONS ===")

print(
    core_no_outcome["category"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n=== VALIDATION ===")

print("Core cases without selected outcomes:",
      len(core_no_outcome))

print("Duplicate case IDs:",
      core_no_outcome["case_id"].duplicated().sum())

print("Missing procedural categories:",
      core_no_outcome["category"].isna().sum())

print("Cases with no decisions:",
      (core_no_outcome["decision_count"] == 0).sum())

print("\n=== PHASE II CASES ===")

print(
    core_no_outcome.loc[
        core_no_outcome["category"]
        == "Phase II initiated - no selected outcome",
        "case_id"
    ].tolist()
)

print("\n=== SAMPLE CASES ===")

display(
    core_no_outcome[
        ["case_id", "case_title",
         "category", "decision_count"]
    ].head(15)
)

assert len(core_no_outcome) == 332
assert core_no_outcome["case_id"].is_unique
assert core_no_outcome["category"].notna().all()

print("\nPASS: All 332 core cases accounted for")


=== CORE NO-OUTCOME CLASSIFICATIONS ===
category
Referral recorded - no selected outcome     278
No recorded decisions                        39
Other procedural decisions only              12
Phase II initiated - no selected outcome      3

=== VALIDATION ===
Core cases without selected outcomes: 332
Duplicate case IDs: 0
Missing procedural categories: 0
Cases with no decisions: 39

=== PHASE II CASES ===
['M.12270', 'M.12236', 'M.11895']

=== SAMPLE CASES ===


,case_id,case_title,category,decision_count
0,M.2845,SOGECABLE / CANALSATÉLITE DIGITAL / VÍA DIGITAL,Referral recorded - no selected outcome,1
1,M.1522,CSME / MSCA / ROCK,Referral recorded - no selected outcome,1
2,M.2857,ECS / IEH,Referral recorded - no selected outcome,1
3,M.1555,HEINEKEN / CRUZCAMPO,Referral recorded - no selected outcome,1
4,M.3737,YARA / AGRO / DLG / JV,Referral recorded - no selected outcome,1
5,M.3758,ALFAMAD / GRUPO FARMACEN,Referral recorded - no selected outcome,1
6,M.2446,GOVIA / CONNEX SOUTH CENTRAL,Referral recorded - no selected outcome,1
7,M.932,SEHB / VIAG / PE - BEWAG,Referral recorded - no selected outcome,1
8,M.3795,INDUSTRI KAPITAL / WIHURI - TRADEKA,Referral recorded - no selected outcome,1
9,M.3318,ECS / SIBELGA,Referral recorded - no selected outcome,1



PASS: All 332 core cases accounted for


In [37]:

# =====================================================
# Stage 2.5 — Task 13
# Investigate 12 Core Procedural-Only Cases
# =====================================================

import pandas as pd

# Identify the 12 procedural-only core cases
procedural_ids = set(
    core_no_outcome.loc[
        core_no_outcome["category"]
        == "Other procedural decisions only",
        "case_id"
    ]
)

# Retrieve case descriptions
case_details = df_cases_core.loc[
    df_cases_core["case_id"].isin(procedural_ids),
    ["case_id", "case_title", "decision_count"]
].copy()

# Retrieve all associated decision records
decision_details = df_decisions_core.loc[
    df_decisions_core["case_id"].isin(procedural_ids),
    [
        "case_id",
        "decision_date",
        "decision_type_labels",
        "decision_category"
    ]
].copy()

# Combine cases with decision records
df_procedural_review = case_details.merge(
    decision_details,
    on="case_id",
    how="left",
    validate="one_to_many"
)

df_procedural_review = df_procedural_review.sort_values(
    ["case_id", "decision_date"]
)

print("=== PROCEDURAL-ONLY CASE SUMMARY ===")

print("Cases investigated:", len(procedural_ids))
print("Decision records:", len(decision_details))

print("\n=== DECISION TYPES ===")

print(
    decision_details["decision_type_labels"]
    .value_counts(dropna=False)
    .to_string()
)

print("\n=== COMPLETE CASE HISTORIES ===")

print(
    df_procedural_review[
        [
            "case_id",
            "case_title",
            "decision_date",
            "decision_type_labels"
        ]
    ].to_string(index=False)
)

print("\n=== VALIDATION ===")

print(
    "Unique cases in review:",
    df_procedural_review["case_id"].nunique()
)

print(
    "Cases missing decision details:",
    df_procedural_review[
        "decision_type_labels"
    ].isna().sum()
)

assert len(procedural_ids) == 12
assert df_procedural_review["case_id"].nunique() == 12
assert df_procedural_review["decision_type_labels"].notna().all()

print("PASS: All 12 cases accounted for")


=== PROCEDURAL-ONLY CASE SUMMARY ===
Cases investigated: 12
Decision records: 15

=== DECISION TYPES ===
decision_type_labels
Art. 7(3)                            6
Art. 22 Refusal of referral          4
Art. 21(4)                           2
Art. 18 (Statement of Objections)    1
Art. 14                              1
2/3 rule                             1

=== COMPLETE CASE HISTORIES ===
case_id                                                    case_title decision_date              decision_type_labels
M.12409                    VANDERLANDE INDUSTRIES / SIEMENS LOGISTICS    2026-05-13       Art. 22 Refusal of referral
 M.3450                        MACQUARIE / MEIF / MCIG / CROWN CASTLE    2004-05-28                         Art. 7(3)
 M.3471                               APAX / CANDOVER / THE TELEGRAPH    2004-05-19                         Art. 7(3)
 M.3488                                      3i / VSS / THE TELEGRAPH    2004-06-09                         Art. 7(3)
 M.3986          

In [38]:

# =====================================================
# Stage 2.5 — Task 14
# Build Consolidated Case-Level Reporting Table
# =====================================================

import pandas as pd
import numpy as np

# Start from the validated core case assessment
df_case_reporting = df_case_outcome_assessment.copy()

# Attach classifications for cases without outcomes
df_case_reporting = df_case_reporting.merge(
    core_no_outcome[
        ["case_id", "category"]
    ].rename(
        columns={"category": "no_outcome_reason"}
    ),
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Identify cases requiring special sequence review
special_ids = set(review_corrected["case_id"])

special_mask = df_case_reporting[
    "case_id"
].isin(special_ids)

no_outcome_mask = df_case_reporting[
    "outcome_record_count"
].eq(0)

multiple_mask = df_case_reporting[
    "outcome_record_count"
].gt(1)

# Preserve the distinction between evidence and outcomes
df_case_reporting["evidence_status"] = np.select(
    [
        special_mask,
        no_outcome_mask,
        multiple_mask
    ],
    [
        "Special sequence review required",
        "No selected outcome",
        "Multiple outcome records - provisional"
    ],
    default="Single outcome record - provisional"
)

# Use latest category only as a provisional candidate
df_case_reporting["candidate_outcome_category"] = (
    df_case_reporting["latest_outcome_category"].copy()
)

# Do not assign candidates to unresolved cases
df_case_reporting.loc[
    special_mask | no_outcome_mask,
    "candidate_outcome_category"
] = pd.NA

# Create a reporting category that retains all cases
df_case_reporting["provisional_reporting_category"] = (
    df_case_reporting[
        "candidate_outcome_category"
    ].astype("string")
)

df_case_reporting.loc[
    no_outcome_mask,
    "provisional_reporting_category"
] = df_case_reporting.loc[
    no_outcome_mask,
    "no_outcome_reason"
]

df_case_reporting.loc[
    special_mask,
    "provisional_reporting_category"
] = "Special sequence - manual verification"

print("=== REPORTING EVIDENCE STATUS ===")

print(
    df_case_reporting["evidence_status"]
    .value_counts()
    .to_string()
)

print("\n=== PROVISIONAL REPORTING CATEGORIES ===")

print(
    df_case_reporting[
        "provisional_reporting_category"
    ].value_counts(dropna=False).to_string()
)

print("\n=== OUTCOME CANDIDATE COVERAGE ===")

print(
    "Cases with provisional outcome candidates:",
    df_case_reporting[
        "candidate_outcome_category"
    ].notna().sum()
)

print(
    "Cases without assigned outcome candidates:",
    df_case_reporting[
        "candidate_outcome_category"
    ].isna().sum()
)

print("\n=== VALIDATION ===")

print("Total core cases:", len(df_case_reporting))

print(
    "Duplicate case IDs:",
    df_case_reporting["case_id"]
    .duplicated().sum()
)

print(
    "Missing reporting categories:",
    df_case_reporting[
        "provisional_reporting_category"
    ].isna().sum()
)

assert len(df_case_reporting) == 10376
assert df_case_reporting["case_id"].is_unique

assert special_mask.sum() == 12
assert no_outcome_mask.sum() == 332

assert (
    df_case_reporting[
        "provisional_reporting_category"
    ].notna().all()
)

print("PASS: Consolidated reporting table validated")


=== REPORTING EVIDENCE STATUS ===
evidence_status
Single outcome record - provisional       9969
No selected outcome                        332
Multiple outcome records - provisional      63
Special sequence review required            12

=== PROVISIONAL REPORTING CATEGORIES ===
provisional_reporting_category
Phase I - Unconditional clearance           9165
Phase I - Conditional clearance              362
Referral recorded - no selected outcome      278
Withdrawal / abandonment                     210
Phase II - Conditional clearance             148
Phase II - Unconditional clearance            66
Outside regulation scope                      50
No recorded decisions                         39
Prohibition                                   31
Special sequence - manual verification        12
Other procedural decisions only               12
Phase II initiated - no selected outcome       3

=== OUTCOME CANDIDATE COVERAGE ===
Cases with provisional outcome candidates: 10032
Cases without as

In [39]:

# =====================================================
# Stage 2.6 — Task 1
# Notification-to-Decision Date Quality Audit
# =====================================================

import pandas as pd
import numpy as np

# Start with the validated case-level reporting table
df_timing_audit = df_case_reporting.merge(
    df_cases_core[["case_id", "notification_dt"]],
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Retain only cases with a provisional outcome candidate
df_timing_audit = df_timing_audit[
    df_timing_audit["candidate_outcome_category"].notna()
].copy()

# Parse dates
df_timing_audit["notification_dt"] = pd.to_datetime(
    df_timing_audit["notification_dt"],
    errors="coerce"
)

df_timing_audit["outcome_dt"] = pd.to_datetime(
    df_timing_audit["latest_outcome_date"],
    errors="coerce"
)

# Calculate observed elapsed calendar days
df_timing_audit["elapsed_days"] = (
    df_timing_audit["outcome_dt"]
    - df_timing_audit["notification_dt"]
).dt.days

# Classify date quality
df_timing_audit["date_quality_status"] = np.select(
    [
        df_timing_audit["notification_dt"].isna(),
        df_timing_audit["outcome_dt"].isna(),
        df_timing_audit["elapsed_days"].lt(0)
    ],
    [
        "Missing notification date",
        "Missing outcome decision date",
        "Negative chronology"
    ],
    default="Non-negative observed interval"
)

print("=== CASE DATE COVERAGE ===")
print(
    "Provisional outcome candidates:",
    len(df_timing_audit)
)

print(
    "Missing notification dates:",
    df_timing_audit["notification_dt"].isna().sum()
)

print(
    "Missing outcome decision dates:",
    df_timing_audit["outcome_dt"].isna().sum()
)

print("\n=== CHRONOLOGY CHECK ===")

print(
    df_timing_audit["date_quality_status"]
    .value_counts()
    .to_string()
)

valid = df_timing_audit[
    df_timing_audit["date_quality_status"]
    == "Non-negative observed interval"
]

print("\n=== LONG OBSERVED INTERVAL FLAGS ===")

print(
    "Intervals exceeding 180 days:",
    (valid["elapsed_days"] > 180).sum()
)

print(
    "Intervals exceeding 365 days:",
    (valid["elapsed_days"] > 365).sum()
)

print(
    "Intervals exceeding 730 days:",
    (valid["elapsed_days"] > 730).sum()
)

print("\n=== DATE QUALITY BY EVIDENCE STATUS ===")

print(
    pd.crosstab(
        df_timing_audit["evidence_status"],
        df_timing_audit["date_quality_status"]
    ).to_string()
)

print("\n=== LONGEST OBSERVED INTERVALS ===")

display(
    valid.sort_values(
        "elapsed_days",
        ascending=False
    )[
        [
            "case_id",
            "case_title",
            "evidence_status",
            "candidate_outcome_category",
            "notification_dt",
            "outcome_dt",
            "elapsed_days"
        ]
    ].head(10)
)

print("\n=== VALIDATION ===")

print(
    "Unique candidate cases:",
    df_timing_audit["case_id"].nunique()
)

print(
    "Duplicate case IDs:",
    df_timing_audit["case_id"].duplicated().sum()
)

assert len(df_timing_audit) == 10032
assert df_timing_audit["case_id"].is_unique

assert (
    df_timing_audit["date_quality_status"]
    .value_counts().sum()
    == 10032
)

print("PASS: All candidate cases assessed")


=== CASE DATE COVERAGE ===
Provisional outcome candidates: 10032
Missing notification dates: 3
Missing outcome decision dates: 0

=== CHRONOLOGY CHECK ===
date_quality_status
Non-negative observed interval    10029
Missing notification date             3

=== LONG OBSERVED INTERVAL FLAGS ===
Intervals exceeding 180 days: 143
Intervals exceeding 365 days: 21
Intervals exceeding 730 days: 8

=== DATE QUALITY BY EVIDENCE STATUS ===
date_quality_status                     Missing notification date  Non-negative observed interval
evidence_status                                                                                  
Multiple outcome records - provisional                          0                              63
Single outcome record - provisional                             3                            9966

=== LONGEST OBSERVED INTERVALS ===


,case_id,case_title,evidence_status,candidate_outcome_category,notification_dt,outcome_dt,elapsed_days
466,M.950,HOFFMANN - LA ROCHE / BOEHRINGER MANNHEIM,Multiple outcome records - provisional,Phase II - Conditional clearance,1997-09-01,2011-05-03,4992.0
2821,M.2567,NORDBANKEN / POSTGIROT,Multiple outcome records - provisional,Phase I - Conditional clearance,2001-09-24,2014-10-13,4767.0
336,M.2876,NEWSCORP / TELEPIU,Multiple outcome records - provisional,Phase II - Conditional clearance,2002-10-16,2010-07-20,2834.0
2145,M.1378,HOECHST / RHONE - POULENC,Multiple outcome records - provisional,Phase I - Conditional clearance,1999-06-24,2004-01-30,1681.0
3942,M.5047,REWE / ADEG,Multiple outcome records - provisional,Phase I - Conditional clearance,2008-04-23,2011-04-29,1101.0
7123,M.7095,SOCAR / DESFA,Single outcome record - provisional,Withdrawal / abandonment,2014-10-01,2017-02-02,855.0
1703,M.269,SHELL / MONTECATINI,Multiple outcome records - provisional,Phase II - Unconditional clearance,1994-01-04,1996-04-24,841.0
6428,M.9343,HYUNDAI HEAVY INDUSTRIES HOLDINGS / DAEWOO SHI...,Single outcome record - provisional,Prohibition,2019-11-12,2022-01-13,793.0
1883,M.2621,SEB / MOULINEX,Multiple outcome records - provisional,Phase II - Unconditional clearance,2001-11-13,2003-11-11,728.0
7711,M.9162,FINCANTIERI / CHANTIERS DE L'ATLANTIQUE,Single outcome record - provisional,Withdrawal / abandonment,2019-09-25,2021-02-02,496.0



=== VALIDATION ===
Unique candidate cases: 10032
Duplicate case IDs: 0
PASS: All candidate cases assessed


In [40]:

# =====================================================
# Stage 2.6 — Task 2
# Review Missing Dates and Long Observed Intervals
# =====================================================

import pandas as pd
import numpy as np

# Preserve the original timing audit
df_timing_review = df_timing_audit.copy()

# Assign mutually exclusive timing review categories
df_timing_review["timing_review_status"] = np.select(
    [
        df_timing_review["notification_dt"].isna(),
        df_timing_review["elapsed_days"] > 730,
        df_timing_review["elapsed_days"] > 365,
        df_timing_review["elapsed_days"] > 180
    ],
    [
        "Missing notification date",
        "Over 730 days",
        "366-730 days",
        "181-365 days"
    ],
    default="0-180 days"
)

print("=== TIMING QUALITY CATEGORIES ===")

print(
    df_timing_review["timing_review_status"]
    .value_counts()
    .to_string()
)

print("\n=== MISSING NOTIFICATION CASES ===")

missing_cases = df_timing_review[
    df_timing_review["notification_dt"].isna()
]

display(
    missing_cases[
        [
            "case_id",
            "case_title",
            "evidence_status",
            "candidate_outcome_category",
            "outcome_dt"
        ]
    ]
)

print("\n=== LONG INTERVALS BY EVIDENCE STATUS ===")

long_cases = df_timing_review[
    df_timing_review["elapsed_days"] > 365
].copy()

print(
    long_cases["evidence_status"]
    .value_counts()
    .to_string()
)

print("\n=== ALL CASES EXCEEDING 365 DAYS ===")

display(
    long_cases[
        [
            "case_id",
            "case_title",
            "evidence_status",
            "outcome_record_count",
            "candidate_outcome_category",
            "elapsed_days"
        ]
    ].sort_values(
        "elapsed_days",
        ascending=False
    )
)

print("\n=== VALIDATION ===")

print(
    "Total candidate cases:",
    len(df_timing_review)
)

print(
    "Missing notification dates:",
    len(missing_cases)
)

print(
    "Cases exceeding 365 days:",
    len(long_cases)
)

print(
    "Duplicate IDs:",
    df_timing_review["case_id"].duplicated().sum()
)

assert len(df_timing_review) == 10032
assert len(missing_cases) == 3
assert len(long_cases) == 21
assert df_timing_review["case_id"].is_unique

print("PASS: Timing review completed")


=== TIMING QUALITY CATEGORIES ===
timing_review_status
0-180 days                   9886
181-365 days                  122
366-730 days                   13
Over 730 days                   8
Missing notification date       3

=== MISSING NOTIFICATION CASES ===


,case_id,case_title,evidence_status,candidate_outcome_category,outcome_dt
90,M.2054,SECIL / HOLDERBANK / CIMPOR,Single outcome record - provisional,Withdrawal / abandonment,2001-01-11
748,M.974,BERTELSMANN / BURDA - HOS T,Single outcome record - provisional,Withdrawal / abandonment,1997-09-26
1990,M.1321,VERBUND / KELAG / PORR / OMV PROTERRA / SIEMEN...,Single outcome record - provisional,Withdrawal / abandonment,1998-12-01



=== LONG INTERVALS BY EVIDENCE STATUS ===
evidence_status
Single outcome record - provisional       11
Multiple outcome records - provisional    10

=== ALL CASES EXCEEDING 365 DAYS ===


,case_id,case_title,evidence_status,outcome_record_count,candidate_outcome_category,elapsed_days
466,M.950,HOFFMANN - LA ROCHE / BOEHRINGER MANNHEIM,Multiple outcome records - provisional,2,Phase II - Conditional clearance,4992.0
2821,M.2567,NORDBANKEN / POSTGIROT,Multiple outcome records - provisional,2,Phase I - Conditional clearance,4767.0
336,M.2876,NEWSCORP / TELEPIU,Multiple outcome records - provisional,2,Phase II - Conditional clearance,2834.0
2145,M.1378,HOECHST / RHONE - POULENC,Multiple outcome records - provisional,2,Phase I - Conditional clearance,1681.0
3942,M.5047,REWE / ADEG,Multiple outcome records - provisional,2,Phase I - Conditional clearance,1101.0
7123,M.7095,SOCAR / DESFA,Single outcome record - provisional,1,Withdrawal / abandonment,855.0
1703,M.269,SHELL / MONTECATINI,Multiple outcome records - provisional,2,Phase II - Unconditional clearance,841.0
6428,M.9343,HYUNDAI HEAVY INDUSTRIES HOLDINGS / DAEWOO SHI...,Single outcome record - provisional,1,Prohibition,793.0
1883,M.2621,SEB / MOULINEX,Multiple outcome records - provisional,2,Phase II - Unconditional clearance,728.0
7711,M.9162,FINCANTIERI / CHANTIERS DE L'ATLANTIQUE,Single outcome record - provisional,1,Withdrawal / abandonment,496.0



=== VALIDATION ===
Total candidate cases: 10032
Missing notification dates: 3
Cases exceeding 365 days: 21
Duplicate IDs: 0
PASS: Timing review completed


In [41]:

# =====================================================
# Stage 2.6 — Task 3
# Review Long Intervals and Zero-Day Observations
# =====================================================

import pandas as pd

# Identify the 21 cases exceeding 365 days
long_cases = df_timing_review.loc[
    df_timing_review["elapsed_days"] > 365
].copy()

long_ids = set(long_cases["case_id"])

# Retrieve ALL recorded decisions for these cases
decision_histories = df_decisions_core.loc[
    df_decisions_core["case_id"].isin(long_ids),
    [
        "case_id",
        "decision_date",
        "decision_type_labels",
        "decision_category"
    ]
].copy()

decision_histories["decision_dt"] = pd.to_datetime(
    decision_histories["decision_date"],
    errors="coerce"
)

# Summarise each case's recorded decision history
history_summary = (
    decision_histories.groupby("case_id")
    .agg(
        total_decision_records=(
            "decision_category", "size"
        ),
        first_recorded_decision=(
            "decision_dt", "min"
        ),
        last_recorded_decision=(
            "decision_dt", "max"
        )
    )
    .reset_index()
)

long_context = long_cases.merge(
    history_summary,
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Check whether decisions exist after the
# selected outcome-related decision
long_context["later_decision_recorded"] = (
    long_context["last_recorded_decision"]
    > long_context["outcome_dt"]
)

print("=== LONG INTERVAL REVIEW SUMMARY ===")

print("Cases exceeding 365 days:", len(long_context))

print(
    "Cases with multiple outcome-related records:",
    (long_context["outcome_record_count"] > 1).sum()
)

print(
    "Cases with one selected outcome-related record:",
    (long_context["outcome_record_count"] == 1).sum()
)

print(
    "Cases with a recorded decision after selected outcome:",
    long_context["later_decision_recorded"].sum()
)

print("\n=== LONG INTERVAL CASE CONTEXT ===")

display(
    long_context[
        [
            "case_id",
            "elapsed_days",
            "outcome_record_count",
            "total_decision_records",
            "candidate_outcome_category",
            "later_decision_recorded"
        ]
    ].sort_values(
        "elapsed_days",
        ascending=False
    )
)

print("\n=== MULTIPLE-OUTCOME DECISION HISTORIES ===")

multi_long_ids = set(
    long_context.loc[
        long_context["outcome_record_count"] > 1,
        "case_id"
    ]
)

display(
    decision_histories.loc[
        decision_histories["case_id"].isin(multi_long_ids),
        [
            "case_id",
            "decision_date",
            "decision_type_labels",
            "decision_category"
        ]
    ].sort_values(
        ["case_id", "decision_date"]
    )
)

# Check zero-day observed intervals separately
zero_day_cases = df_timing_review.loc[
    df_timing_review["elapsed_days"] == 0
].copy()

print("\n=== ZERO-DAY INTERVAL CHECK ===")

print("Cases with zero-day intervals:", len(zero_day_cases))

print(
    zero_day_cases["candidate_outcome_category"]
    .value_counts()
    .to_string()
)

print("\n=== ZERO-DAY EXAMPLES ===")

display(
    zero_day_cases[
        [
            "case_id",
            "notification_dt",
            "outcome_dt",
            "candidate_outcome_category"
        ]
    ].head(10)
)

print("\n=== VALIDATION ===")

print("Total candidate cases:", len(df_timing_review))
print("Long-interval cases reviewed:", len(long_context))
print("Duplicate long-interval cases:",
      long_context["case_id"].duplicated().sum())

assert len(df_timing_review) == 10032
assert len(long_context) == 21
assert long_context["case_id"].is_unique
assert long_context["total_decision_records"].notna().all()

print("PASS: Timing exception investigation completed")


=== LONG INTERVAL REVIEW SUMMARY ===
Cases exceeding 365 days: 21
Cases with multiple outcome-related records: 10
Cases with one selected outcome-related record: 11
Cases with a recorded decision after selected outcome: 5

=== LONG INTERVAL CASE CONTEXT ===


,case_id,elapsed_days,outcome_record_count,total_decision_records,candidate_outcome_category,later_decision_recorded
1,M.950,4992.0,2,3,Phase II - Conditional clearance,False
6,M.2567,4767.0,2,2,Phase I - Conditional clearance,False
0,M.2876,2834.0,2,5,Phase II - Conditional clearance,False
5,M.1378,1681.0,2,2,Phase I - Conditional clearance,False
8,M.5047,1101.0,2,2,Phase I - Conditional clearance,False
13,M.7095,855.0,1,2,Withdrawal / abandonment,False
3,M.269,841.0,2,3,Phase II - Unconditional clearance,False
12,M.9343,793.0,1,2,Prohibition,False
4,M.2621,728.0,2,4,Phase II - Unconditional clearance,False
14,M.9162,496.0,1,4,Withdrawal / abandonment,False



=== MULTIPLE-OUTCOME DECISION HISTORIES ===


,case_id,decision_date,decision_type_labels,decision_category
10050,M.10507,2022-11-03,Withdrawn,Withdrawal / abandonment
10049,M.10507,2023-10-30,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
10048,M.10507,2024-04-03,Purchaser approval,Other procedural decision
2385,M.1378,1999-08-09,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
2386,M.1378,2004-01-30,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
3147,M.2567,2001-11-08,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
3146,M.2567,2014-10-13,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
2098,M.2621,2002-01-08,Art. 9(3) partial referral,Other procedural decision
2099,M.2621,2002-01-08,Art. 6(1)(b) with conditions & obligations,Phase I - Conditional clearance
2097,M.2621,2003-05-23,Art. 6(1)(c),Phase II initiated



=== ZERO-DAY INTERVAL CHECK ===
Cases with zero-day intervals: 2
candidate_outcome_category
Withdrawal / abandonment             1
Phase I - Unconditional clearance    1

=== ZERO-DAY EXAMPLES ===


,case_id,notification_dt,outcome_dt,candidate_outcome_category
82,M.405,1994-04-05,1994-04-05,Withdrawal / abandonment
7014,M.7046,2013-10-11,2013-10-11,Phase I - Unconditional clearance



=== VALIDATION ===
Total candidate cases: 10032
Long-interval cases reviewed: 21
Duplicate long-interval cases: 0
PASS: Timing exception investigation completed


In [42]:

# =====================================================
# Stage 2.6 — Task 4
# Final Targeted Timing Quality Investigation
# =====================================================

import pandas as pd

# Five cases with decisions after selected outcome
later_ids = set(
    long_context.loc[
        long_context["later_decision_recorded"],
        "case_id"
    ]
)

# Two cases with zero-day intervals
zero_ids = set(
    df_timing_review.loc[
        df_timing_review["elapsed_days"] == 0,
        "case_id"
    ]
)

review_ids = later_ids | zero_ids

print("=== FINAL TIMING EXCEPTION SUMMARY ===")
print("Post-outcome decision cases:", len(later_ids))
print("Zero-day cases:", len(zero_ids))
print("Unique cases for review:", len(review_ids))

# Retrieve full case information
case_info = df_timing_review.loc[
    df_timing_review["case_id"].isin(review_ids),
    [
        "case_id",
        "case_title",
        "notification_dt",
        "outcome_dt",
        "elapsed_days",
        "candidate_outcome_category"
    ]
].copy()

# Retrieve all decision records
review_decisions = df_decisions_core.loc[
    df_decisions_core["case_id"].isin(review_ids),
    [
        "case_id",
        "decision_date",
        "decision_type_labels",
        "decision_category"
    ]
].copy()

review_decisions["decision_dt"] = pd.to_datetime(
    review_decisions["decision_date"],
    errors="coerce"
)

# Attach notification and selected outcome dates
review_decisions = review_decisions.merge(
    case_info[
        ["case_id", "notification_dt", "outcome_dt"]
    ],
    on="case_id",
    how="left",
    validate="many_to_one"
)

# Indicate where each decision sits chronologically
review_decisions["decision_position"] = (
    "Before or on selected outcome"
)

review_decisions.loc[
    review_decisions["decision_dt"]
    > review_decisions["outcome_dt"],
    "decision_position"
] = "After selected outcome"

print("\n=== CASE TIMING SUMMARY ===")

display(
    case_info.sort_values("case_id")
)

print("\n=== COMPLETE DECISION HISTORIES ===")

print(
    review_decisions.sort_values(
        ["case_id", "decision_dt"]
    )[
        [
            "case_id",
            "decision_date",
            "decision_type_labels",
            "decision_category",
            "decision_position"
        ]
    ].to_string(index=False)
)

print("\n=== LATER DECISION TYPES ===")

print(
    review_decisions.loc[
        review_decisions["decision_position"]
        == "After selected outcome",
        "decision_type_labels"
    ].value_counts().to_string()
)

print("\n=== VALIDATION ===")

print("Unique cases reviewed:", len(review_ids))

print(
    "Cases with decision histories:",
    review_decisions["case_id"].nunique()
)

print(
    "Missing decision dates:",
    review_decisions["decision_dt"].isna().sum()
)

assert len(later_ids) == 5
assert len(zero_ids) == 2
assert review_decisions["case_id"].nunique() == len(review_ids)
assert review_decisions["decision_dt"].notna().all()

print("PASS: Final timing exceptions investigated")


=== FINAL TIMING EXCEPTION SUMMARY ===
Post-outcome decision cases: 5
Zero-day cases: 2
Unique cases for review: 7

=== CASE TIMING SUMMARY ===


,case_id,case_title,notification_dt,outcome_dt,elapsed_days,candidate_outcome_category
8761,M.10149,KOREAN AIR LINES / ASIANA AIRLINES,2023-01-13,2024-02-13,396.0,Phase II - Conditional clearance
8678,M.10188,ILLUMINA / GRAIL,2021-06-16,2022-09-06,447.0,Prohibition
8921,M.10507,HITACHI RAIL / GROUND TRANSPORTATION SYSTEMS B...,2022-10-04,2023-10-30,391.0,Phase I - Conditional clearance
82,M.405,GE / NUOVO PIGNONE,1994-04-05,1994-04-05,0.0,Withdrawal / abandonment
7014,M.7046,PARKWIND / SUMMIT RENEWABLE ENERGY BELWIND 1 /...,2013-10-11,2013-10-11,0.0,Phase I - Unconditional clearance
5894,M.9014,PKN ORLEN / GRUPA LOTOS,2019-07-03,2020-07-14,377.0,Phase II - Conditional clearance
5779,M.9569,ESSILORLUXOTTICA / GRANDVISION,2019-12-23,2021-03-23,456.0,Phase II - Conditional clearance



=== COMPLETE DECISION HISTORIES ===
case_id decision_date                       decision_type_labels                 decision_category             decision_position
M.10149    2023-02-17                               Art. 6(1)(c)                Phase II initiated Before or on selected outcome
M.10149    2023-05-17          Art. 18 (Statement of Objections)         Other procedural decision Before or on selected outcome
M.10149    2024-02-13    Art. 8(2) with conditions & obligations  Phase II - Conditional clearance Before or on selected outcome
M.10149    2024-11-15 Decision on the implementation of remedies         Other procedural decision        After selected outcome
M.10149    2024-11-28                         Purchaser approval         Other procedural decision        After selected outcome
M.10188    2021-04-19                      Art. 22 Full referral         Other procedural decision Before or on selected outcome
M.10188    2021-07-22                               Art. 6(1

In [43]:

# =====================================================
# Stage 2.6 — Task 5
# Consolidated Data Quality Exception Register
# =====================================================

import pandas as pd

# Preserve the validated reporting table
df_core_quality = df_case_reporting.copy()

# Attach previously audited timing information
df_core_quality = df_core_quality.merge(
    df_timing_review[
        ["case_id", "notification_dt",
         "outcome_dt", "elapsed_days"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Identify formal withdrawal-of-decision records
# These differ from transaction withdrawal events
withdrawal_mask = (
    df_decisions_core["decision_type_labels"]
    .fillna("")
    .astype(str)
    .str.contains(
        r"^\s*Withdrawal of Art\.",
        case=False,
        regex=True
    )
)

df_formal_withdrawals = df_decisions_core.loc[
    withdrawal_mask,
    ["case_id", "decision_date", "decision_type_labels"]
].copy()

formal_withdrawal_ids = set(
    df_formal_withdrawals["case_id"]
)

# Consolidate previously established quality flags
flags = {
    "no_selected_outcome":
        df_core_quality["outcome_record_count"].eq(0),

    "special_sequence":
        df_core_quality["case_id"].isin(
            set(review_corrected["case_id"])
        ),

    "multiple_outcome_records":
        df_core_quality["outcome_record_count"].gt(1),

    "missing_notification_for_candidate":
        df_core_quality["candidate_outcome_category"].notna()
        & df_core_quality["notification_dt"].isna(),

    "observed_interval_over_365":
        df_core_quality["elapsed_days"].gt(365),

    "zero_day_interval":
        df_core_quality["elapsed_days"].eq(0),

    "later_procedural_decision":
        df_core_quality["case_id"].isin(later_ids),

    "formal_decision_withdrawal_record":
        df_core_quality["case_id"].isin(
            formal_withdrawal_ids
        ),

    "verified_withdrawn_prohibition":
        df_core_quality["case_id"].eq("M.10188"),

    "verified_decision_date_discrepancy":
        df_core_quality["case_id"].eq("M.7046")
}

for name, condition in flags.items():
    df_core_quality[name] = condition.fillna(False)

# Flags may overlap. Do not sum them as unique cases.
print("=== CONSOLIDATED QUALITY FLAGS ===")

for name in flags:
    print(
        f"{name}:",
        int(df_core_quality[name].sum())
    )

print("\n=== FORMAL DECISION WITHDRAWAL RECORDS ===")

print(
    df_formal_withdrawals.sort_values(
        ["case_id", "decision_date"]
    ).to_string(index=False)
)

print("\n=== SOURCE-VERIFIED EXCEPTIONS ===")

display(
    df_core_quality.loc[
        df_core_quality["case_id"].isin(
            ["M.10188", "M.7046"]
        ),
        [
            "case_id",
            "candidate_outcome_category",
            "notification_dt",
            "outcome_dt",
            "elapsed_days",
            "verified_withdrawn_prohibition",
            "verified_decision_date_discrepancy"
        ]
    ]
)

print("\n=== FINAL RECONCILIATION ===")

print("Core merger cases:", len(df_core_quality))
print(
    "Unique case IDs:",
    df_core_quality["case_id"].nunique()
)
print(
    "Duplicate case IDs:",
    df_core_quality["case_id"].duplicated().sum()
)

assert len(df_core_quality) == 10376
assert df_core_quality["case_id"].is_unique
assert df_core_quality[
    "no_selected_outcome"
].sum() == 332
assert df_core_quality[
    "special_sequence"
].sum() == 12
assert df_core_quality[
    "missing_notification_for_candidate"
].sum() == 3
assert df_core_quality[
    "observed_interval_over_365"
].sum() == 21
assert df_core_quality[
    "zero_day_interval"
].sum() == 2
assert df_core_quality[
    "later_procedural_decision"
].sum() == 5

print("PASS: Consolidated quality register validated")


=== CONSOLIDATED QUALITY FLAGS ===
no_selected_outcome: 332
special_sequence: 12
multiple_outcome_records: 75
missing_notification_for_candidate: 3
observed_interval_over_365: 21
zero_day_interval: 2
later_procedural_decision: 5
formal_decision_withdrawal_record: 1
verified_withdrawn_prohibition: 1
verified_decision_date_discrepancy: 1

=== FORMAL DECISION WITHDRAWAL RECORDS ===
case_id decision_date                decision_type_labels
M.10188    2024-09-06 Withdrawal of Art. 6(1)(c) decision
M.10188    2024-09-06    Withdrawal of Art. 8(3) decision

=== SOURCE-VERIFIED EXCEPTIONS ===


,case_id,candidate_outcome_category,notification_dt,outcome_dt,elapsed_days,verified_withdrawn_prohibition,verified_decision_date_discrepancy
7014,M.7046,Phase I - Unconditional clearance,2013-10-11,2013-10-11,0.0,False,True
8678,M.10188,Prohibition,2021-06-16,2022-09-06,447.0,True,False



=== FINAL RECONCILIATION ===
Core merger cases: 10376
Unique case IDs: 10376
Duplicate case IDs: 0
PASS: Consolidated quality register validated


In [44]:

# ==========================================================
# STAGE 2.8 — FINAL DATA CLEANING
# Prepare Core Merger Tables for SQL
# ==========================================================

import pandas as pd
import numpy as np

# --- 1. Build final case table ---

df_cases_sql = df_core_quality.copy()

source_columns = [
    "case_id",
    "case_number",
    "regulation",
    "notification_date",
    "last_recorded_decision_date",
    "decision_count",
    "sector_count",
    "date_quality_flag",
    "chronology_flag"
]

df_cases_sql = df_cases_sql.merge(
    df_cases_core[source_columns],
    on="case_id",
    how="left",
    validate="one_to_one"
)

# Preserve original outcome information
df_cases_sql["source_outcome_category"] = (
    df_cases_sql["candidate_outcome_category"]
)

df_cases_sql["source_outcome_date"] = pd.to_datetime(
    df_cases_sql["latest_outcome_date"],
    errors="coerce"
)

# --- 2. Apply conservative outcome eligibility ---

# Strict category analysis uses only single-outcome cases
# and excludes a formally withdrawn prohibition.
strict_outcome = (
    df_cases_sql["evidence_status"].eq(
        "Single outcome record - provisional"
    )
    & df_cases_sql["source_outcome_category"].notna()
    & ~df_cases_sql["verified_withdrawn_prohibition"]
)

df_cases_sql["outcome_kpi_eligible"] = strict_outcome

df_cases_sql["outcome_category_for_sql"] = (
    df_cases_sql["source_outcome_category"]
    .where(strict_outcome, pd.NA)
)

# Preserve multi-outcome cases as a separate review population
df_cases_sql["outcome_review_required"] = (
    df_cases_sql["multiple_outcome_records"]
    | df_cases_sql["no_selected_outcome"]
    | df_cases_sql["verified_withdrawn_prohibition"]
)

# --- 3. Apply traceable date correction ---

df_cases_sql["analytical_outcome_date"] = (
    df_cases_sql["source_outcome_date"].copy()
)

df_cases_sql["date_correction_applied"] = False

# M.7046: Official Commission decision and OJ support
# 2013-11-11; original extract records 2013-10-11.
m7046 = df_cases_sql["case_id"].eq("M.7046")

assert m7046.sum() == 1
assert (
    df_cases_sql.loc[
        m7046, "source_outcome_date"
    ].iloc[0] == pd.Timestamp("2013-10-11")
)

df_cases_sql.loc[
    m7046, "analytical_outcome_date"
] = pd.Timestamp("2013-11-11")

df_cases_sql.loc[
    m7046, "date_correction_applied"
] = True

# --- 4. Prepare notification and interval fields ---

df_cases_sql["notification_dt"] = pd.to_datetime(
    df_cases_sql["notification_dt"],
    errors="coerce"
)

df_cases_sql["notification_year"] = (
    df_cases_sql["notification_dt"].dt.year.astype("Int64")
)

df_cases_sql["analytical_interval_days"] = (
    df_cases_sql["analytical_outcome_date"]
    - df_cases_sql["notification_dt"]
).dt.days.astype("Int64")

# Keep original source interval for comparison
df_cases_sql["source_interval_days"] = (
    df_cases_sql["elapsed_days"]
)

# --- 5. Identify Phase I and Phase II decision groups ---

phase1 = {
    "Phase I - Unconditional clearance",
    "Phase I - Conditional clearance"
}

phase2 = {
    "Phase II - Unconditional clearance",
    "Phase II - Conditional clearance",
    "Prohibition"
}

df_cases_sql["timing_phase"] = np.select(
    [
        df_cases_sql["source_outcome_category"].isin(phase1),
        df_cases_sql["source_outcome_category"].isin(phase2)
    ],
    ["Phase I", "Phase II"],
    default="Other / unresolved"
)

# Use strict single-outcome cases for the primary timing
# proxy. Do not automatically remove long valid intervals.
df_cases_sql["timing_kpi_eligible"] = (
    df_cases_sql["outcome_kpi_eligible"]
    & df_cases_sql["timing_phase"].isin(
        ["Phase I", "Phase II"]
    )
    & df_cases_sql["notification_dt"].notna()
    & df_cases_sql["analytical_outcome_date"].notna()
    & df_cases_sql["analytical_interval_days"].ge(0)
)

# --- 6. Add historical Phase II initiation evidence ---

phase2_ids = set(
    df_decisions_core.loc[
        df_decisions_core["decision_type_labels"]
        .astype(str).str.strip()
        .eq("Art. 6(1)(c)"),
        "case_id"
    ]
)

df_cases_sql["phase_ii_initiated_recorded"] = (
    df_cases_sql["case_id"].isin(phase2_ids)
)

# This flag does NOT establish the final KPI 2 denominator.
df_cases_sql["has_notification_date"] = (
    df_cases_sql["notification_dt"].notna()
)

df_cases_sql["has_recorded_decision"] = (
    df_cases_sql["decision_count"].gt(0)
)

df_cases_sql["has_sector_association"] = (
    df_cases_sql["sector_count"].gt(0)
)

# --- 7. Prepare decision table ---

df_decisions_sql = df_decisions_core.copy()

df_decisions_sql["decision_date_analytical"] = (
    pd.to_datetime(
        df_decisions_sql["decision_date"],
        errors="coerce"
    )
)

decision_m7046 = df_decisions_sql["case_id"].eq("M.7046")

df_decisions_sql["date_correction_applied"] = False

df_decisions_sql.loc[
    decision_m7046, "decision_date_analytical"
] = pd.Timestamp("2013-11-11")

df_decisions_sql.loc[
    decision_m7046, "date_correction_applied"
] = True

# Preserve historical M.10188 events, mark withdrawals
df_decisions_sql["subsequently_withdrawn_decision"] = (
    df_decisions_sql["case_id"].eq("M.10188")
    & df_decisions_sql["decision_type_labels"].isin(
        ["Art. 6(1)(c)", "Art. 8(3)"]
    )
)

# --- 8. Preserve all validated sector relationships ---

df_sectors_sql = df_sectors_core.copy()

# --- 9. Final analytical checks ---

assert len(df_cases_sql) == 10376
assert len(df_decisions_sql) == 11579
assert len(df_sectors_sql) == 16286

assert df_cases_sql["case_id"].is_unique

assert not (
    set(df_decisions_sql["case_id"])
    - set(df_cases_sql["case_id"])
)

assert not (
    set(df_sectors_sql["case_id"])
    - set(df_cases_sql["case_id"])
)

assert df_cases_sql["outcome_kpi_eligible"].sum() == 9968

assert not df_cases_sql.loc[
    df_cases_sql["case_id"].eq("M.10188"),
    "outcome_kpi_eligible"
].any()

assert (
    df_cases_sql.loc[
        m7046, "analytical_interval_days"
    ].iloc[0] == 31
)

assert df_decisions_sql.loc[
    decision_m7046, "date_correction_applied"
].all()

print("=== FINAL CLEANING SUMMARY ===")
print("Core cases:", len(df_cases_sql))
print("Core decisions:", len(df_decisions_sql))
print("Core sector relationships:", len(df_sectors_sql))
print(
    "Strict provisional outcome-eligible cases:",
    int(df_cases_sql["outcome_kpi_eligible"].sum())
)
print(
    "Cases requiring outcome review:",
    int(df_cases_sql["outcome_review_required"].sum())
)
print(
    "Timing-proxy eligible cases:",
    int(df_cases_sql["timing_kpi_eligible"].sum())
)
print(
    "Recorded Phase II initiation cases:",
    int(df_cases_sql["phase_ii_initiated_recorded"].sum())
)
print("PASS: Final cleaning rules applied")


=== FINAL CLEANING SUMMARY ===
Core cases: 10376
Core decisions: 11579
Core sector relationships: 16286
Strict provisional outcome-eligible cases: 9968
Cases requiring outcome review: 408
Timing-proxy eligible cases: 9708
Recorded Phase II initiation cases: 309
PASS: Final cleaning rules applied


In [45]:

# ==========================================================
# STAGE 2.8 — EXPORT AND ROUND-TRIP VALIDATION
# ==========================================================

from pathlib import Path
import json
import zipfile
import pandas as pd

output_dir = Path("/content/project5_sql_ready")
output_dir.mkdir(parents=True, exist_ok=True)

# Document source-grounded changes
df_cleaning_log = pd.DataFrame([
    {
        "case_id": "M.7046",
        "issue": "Source decision-date discrepancy",
        "original_value": "2013-10-11",
        "analytical_value": "2013-11-11",
        "treatment": "Correct derived date; retain source date",
        "source_url":
        "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:C:2013:339:FULL"
    },
    {
        "case_id": "M.10188",
        "issue": "Historical prohibition subsequently withdrawn",
        "original_value": "Prohibition",
        "analytical_value": "Excluded from strict outcome KPI",
        "treatment": "Retain decision history and withdrawal flag",
        "source_url":
        "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=CELEX:52024M10188"
    },
    {
        "case_id": "",
        "issue": "Multiple or unresolved outcomes",
        "original_value": "75 multiple / 332 no selected outcome",
        "analytical_value": "Separate review population",
        "treatment": "Retain cases; strict KPI eligibility flag",
        "source_url": "Internal validated data-quality audit"
    },
    {
        "case_id": "",
        "issue": "Incomplete or unusual timing evidence",
        "original_value": "3 missing dates; 21 intervals >365 days",
        "analytical_value": "Missing values and flags preserved",
        "treatment": "No automatic imputation or outlier removal",
        "source_url": "Internal validated timing audit"
    }
])

# Exportable copies: avoid modifying notebook tables
tables = {
    "cases_core_sql_ready.csv": df_cases_sql.copy(),
    "decisions_core_sql_ready.csv": df_decisions_sql.copy(),
    "sectors_core_sql_ready.csv": df_sectors_sql.copy(),
    "cases_supplementary.csv":
        df_cases_supplementary.copy(),
    "decisions_supplementary.csv":
        df_decisions_supplementary.copy(),
    "sectors_supplementary.csv":
        df_sectors_supplementary.copy(),
    "cleaning_change_log.csv": df_cleaning_log.copy()
}

# Make list/dict cells safe for CSV
def csv_safe(value):
    if isinstance(value, (list, dict, tuple)):
        return json.dumps(value, ensure_ascii=False)
    return value

for filename, frame in tables.items():
    for column in frame.columns:
        if frame[column].dtype == "object":
            frame[column] = frame[column].map(csv_safe)

    # MySQL-friendly 0/1 flags
    bool_columns = frame.select_dtypes(
        include=["bool"]
    ).columns

    for column in bool_columns:
        frame[column] = frame[column].astype("int8")

    frame.to_csv(
        output_dir / filename,
        index=False,
        encoding="utf-8",
        date_format="%Y-%m-%d"
    )

print("=== CSV EXPORT SUMMARY ===")

for filename in tables:
    print(filename, "|", len(tables[filename]), "rows")

# Read the exported core files back from disk
cases_check = pd.read_csv(
    output_dir / "cases_core_sql_ready.csv",
    dtype={"case_id": "string"}
)

decisions_check = pd.read_csv(
    output_dir / "decisions_core_sql_ready.csv",
    dtype={"case_id": "string"}
)

sectors_check = pd.read_csv(
    output_dir / "sectors_core_sql_ready.csv",
    dtype={
        "case_id": "string",
        "sector_code": "string"
    }
)

print("\n=== CSV ROUND-TRIP VALIDATION ===")

assert len(cases_check) == 10376
assert len(decisions_check) == 11579
assert len(sectors_check) == 16286

assert cases_check["case_id"].is_unique

assert not decisions_check.duplicated(
    ["case_id", "decision_id"]
).any()

assert not sectors_check.duplicated(
    ["case_id", "sector_code"]
).any()

assert set(decisions_check["case_id"]).issubset(
    set(cases_check["case_id"])
)

assert set(sectors_check["case_id"]).issubset(
    set(cases_check["case_id"])
)

assert cases_check["outcome_kpi_eligible"].sum() == 9968

# Check source record reconciliation
assert (
    len(cases_check) + len(df_cases_supplementary)
    == 10404
)

assert (
    len(decisions_check) + len(df_decisions_supplementary)
    == 11625
)

assert (
    len(sectors_check) + len(df_sectors_supplementary)
    == 16310
)

# Check decision date correction survived export
assert (
    cases_check.loc[
        cases_check["case_id"].eq("M.7046"),
        "analytical_interval_days"
    ].iloc[0] == 31
)

print("PASS: All exported tables validated")

# Build one archive for convenient storage
zip_path = Path("/content/Project5_SQL_Ready.zip")

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for path in output_dir.glob("*.csv"):
        archive.write(path, arcname=path.name)

print("\n=== FINAL OUTPUT ===")
print("Archive:", zip_path)
print("READY FOR SQL IMPORT")


=== CSV EXPORT SUMMARY ===
cases_core_sql_ready.csv | 10376 rows
decisions_core_sql_ready.csv | 11579 rows
sectors_core_sql_ready.csv | 16286 rows
cases_supplementary.csv | 28 rows
decisions_supplementary.csv | 46 rows
sectors_supplementary.csv | 24 rows
cleaning_change_log.csv | 4 rows

=== CSV ROUND-TRIP VALIDATION ===
PASS: All exported tables validated

=== FINAL OUTPUT ===
Archive: /content/Project5_SQL_Ready.zip
READY FOR SQL IMPORT


In [46]:

# Stage 2.8 — Final correction before SQL import
# Restore notification dates for all core merger cases

from pathlib import Path
import pandas as pd
import zipfile

original_dates = (
    df_cases_core.set_index("case_id")["notification_dt"]
)

before = df_cases_sql["notification_dt"].notna().sum()

df_cases_sql["notification_dt"] = pd.to_datetime(
    df_cases_sql["case_id"].map(original_dates),
    errors="coerce"
)

df_cases_sql["notification_year"] = (
    df_cases_sql["notification_dt"].dt.year.astype("Int64")
)

df_cases_sql["has_notification_date"] = (
    df_cases_sql["notification_dt"].notna()
)

df_cases_sql["analytical_interval_days"] = (
    df_cases_sql["analytical_outcome_date"]
    - df_cases_sql["notification_dt"]
).dt.days.astype("Int64")

expected = df_cases_core["notification_dt"].notna().sum()
actual = df_cases_sql["notification_dt"].notna().sum()

assert actual == expected
assert len(df_cases_sql) == 10376

# Re-export the corrected cases table
output_dir = Path("/content/project5_sql_ready")
output_dir.mkdir(parents=True, exist_ok=True)

export_cases = df_cases_sql.copy()

for col in export_cases.select_dtypes(
    include=["bool"]
).columns:
    export_cases[col] = export_cases[col].astype("int8")

csv_path = output_dir / "cases_core_sql_ready.csv"

export_cases.to_csv(
    csv_path,
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)

# Confirm the corrected CSV contents
check = pd.read_csv(csv_path)

assert len(check) == 10376
assert check["notification_dt"].notna().sum() == expected
assert check["has_notification_date"].sum() == expected

# Rebuild the existing ZIP
zip_path = Path("/content/Project5_SQL_Ready.zip")

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as archive:
    for file in output_dir.glob("*.csv"):
        archive.write(file, arcname=file.name)

print("Previous notification dates populated:", before)
print("Corrected notification dates populated:", actual)
print("Notification dates restored:", actual - before)
print("PASS: Corrected CSV and ZIP validated")


Previous notification dates populated: 10029
Corrected notification dates populated: 10131
Notification dates restored: 102
PASS: Corrected CSV and ZIP validated


In [47]:
from google.colab import files
files.download("/content/Project5_SQL_Ready.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>